In [ ]:
import nibabel
from nibabel import cifti2
import numpy as np
import scipy.io as sio
import glob, os
import matplotlib.pyplot as plt
import nibabel as nib
import pandas as pd
import matplotlib
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.decomposition import PCA
import scipy.stats
import statsmodels.api as sm
from patsy import dmatrices
import nilearn
from sklearn.decomposition import FastICA

from scipy import signal
from scipy.signal import butter, filtfilt
import numpy as np
import matplotlib.pyplot as plt
import os, glob, shutil
import nibabel as nib

import statsmodels.api as sm
from patsy import dmatrices

### Overview of this notebook

This notebook contains the key functions that were used in the paper "Idiosyncrasy of fMRI signals in at-risk adolescents: relationships with suicide risk factors in a longitudinal study."

Relevant details:
- This notebook assumes imaging data has been parcellated with the Schaefer/Yeo 400 Region (and 17 network) atlas.
- All permutation tests are set to a low number of iterations because the same functions that run the permutations also produce other plots and statistics. For actual analyses please set to at least the number of iterations to 1000 or so. 
- There are a number of outputs from this notebook that were used for intermediate analysis and not for final publication.
- The behavioral and imaging data that is necessary to fully run this notebook are not provided in this repository. Therefore the goal of this notebook is to provide transparency into the methods that were used for analyses.
- Apologies that this notebook is not cleaner! The project ended quite a while before publication and limited time prevents me from doing further cleaning.


### Load Reference Image to Get Region Names

In [ ]:
cifti_reference_path = ''
cifti_output_folder = ''
input_dir = ''
nuisance_files_dir = ''
csv_path = ''
mixed_effects_output_dir_template = ''

In [ ]:
parc = nib.load(cifti_reference_path)
regions = parc.header.get_axis(1).__dict__['name']

### A couple of preliminary functions

In [ ]:
def make_pscalar_cifti(data_array, template_cifti_path, output_cifti_path, dimension_names=None):
    """
    #Function that creates a cifti pscalar.nii file for
    #visualizing overlays. data_array should  have a shape
    #<n_dimensions, n_regions> where n_dimensions is at least
    #1. template_cifti_path is the path to a cifti file (either
    #pscalar or ptseries works) that has the same parcel definitions
    #as desired for visualizing the data_array. Output_cifti_path
    #specifies the name of the file to be saved (should end in
    #*.pscalar.nii). dimension_names is optional list that will serve
    #as the name of different dimensions (i.e. PC1, PC2, etc.). If
    #specified, dimension_names should have one element for each dimension
    #(not region) in data_array. Function saves new file to output path,
    #doesn't return anything
    """
    
    #Set dimension names if not specified
    if type(dimension_names) == type(None):
        dimension_names = ['NONAME']*data_array.shape[0]
    
    #Load template cifti file
    template_cifti = nib.load(template_cifti_path)
    
    #Copy/generate axis for new cifti file
    pscalar_axis_1 = template_cifti.header.get_axis(1)
    pscalar_axis_0 = nib.cifti2.cifti2_axes.ScalarAxis(dimension_names)
    
    #Put the axes into a header
    cifti_hdr = nib.cifti2.cifti2_axes.to_header([pscalar_axis_0, pscalar_axis_1])
    
    #put the data array and axes into a new image
    cifti_image = nib.cifti2.cifti2.Cifti2Image(dataobj=data_array, header=cifti_hdr)
    
    #Save the cifti image
    nib.save(cifti_image, output_cifti_path)
    
    return

def imagesc_schaeffer_17(connectivity_matrix, parcel_labels, minmax, border_width=5, add_colorbar=True, dpi=200,
                         x_tick_labels=True, y_tick_labels=True, matplotlib_color_scheme='jet',
                         x_tick_font_size='xx-small',y_tick_font_size='xx-small', title='', linewidth = 1, linecolor='black',
                         exclude_no_network_rois = True):

    """This function can make a connectomic plot for the 17 network schaeffer parcellation
    at any resolution. Needs to take a nxn numpy matrix, a length n list of parcel names (taken
    directly from the Schaeffer/Yeo parcellation), and a two element list specifying the
    minimum and maximum for the color scale (i.e. minmax = [0, 1]). You can choose how
    wide you want the border coloring to be, whether or not to use a colorbar, figure resolution,
    etc. (see kwargs above). tick font size and coloring schemes accept values that work with
    matplotlib.
    
    exclude_no_network_rois: bool
        If False, the function will throw an error if there
        are any ROIs without networks. If True, this function
        may attempt to remove these.


    example usage:
    imagesc_schaeffer_17(nxn_conn_mat_as_np_array, len_n_list_of_label_names, [-1, 1])"""


    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    #The names of the different networks for the visualization
    network_names = ['Vis. A', 'Vis. B', 'SomMot. A', 'SomMot. B', 'Temp. Par.', 'Dors. Attn. A',
                    'Dors. Attn. B', 'Sal. A', 'Sal. B', 'Cont. A', 'Cont. B', 'Cont. C', 'DMN A',
                    'DMN B', 'DMN C', 'Limbic A', 'Limbic B']

    #The name of different networks to pull out of the labels
    network_identifiers = ['VisCent','VisPeri','SomMotA','SomMotB','TempPar', 'DorsAttnA',
                           'DorsAttnB','SalVentAttnA','SalVentAttnB','ContA','ContB','ContC','DefaultA',
                           'DefaultB','DefaultC','Limbic_TempPole', 'Limbic_OFC']

    alt_network_identifiers = ['VisCent','VisPeri','SomMotA','SomMotB','TempPar', 'DorsAttnA',
                       'DorsAttnB','SalVentAttnA','SalVentAttnB','ContA','ContB','ContC','DefaultA',
                       'DefaultB','DefaultC','LimbicA','LimbicB']

    network_colors = [
        [97/255, 38/255, 107/255, 1], #vis. a
        [195/255, 40/255, 39/255, 1], #vis. b
        [79/255, 130/255, 165/255, 1], #sommot a
        [82/255, 181/255, 140/255, 1], #sommat b
        [53/255, 75/255, 159/255, 1], #temp par
        [75/255, 147/255, 72/255, 1], #dors attn a
        [50/255, 116/255, 62/255, 1], #dors attn b
        [149/255, 77/255, 158/255, 1], #sal A
        [222/255, 130/255, 177/255, 1], #sal B
        [210/255, 135/255, 48/255, 1], #cont a
        [132/255, 48/255, 73/255, 1], #cont b
        [92/255, 107/255, 130/255, 1], #cont c
        [217/255, 221/255, 72/255, 1], #dmn a
        [176/255, 49/255, 69/255, 1], #dmn b
        [41/255, 37/255, 99/255, 1], #dmn c
        [75/255, 87/255, 61/255, 1], #limbic a
        [149/255, 166/255, 110/255, 1] #limbic b
    ]


    #Array to store network IDs (0-N, corresponding to order of network names)
    network_ids = np.zeros(len(parcel_labels))
    all_networks_present = np.zeros(len(network_identifiers))
    no_network_inds = np.zeros(len(parcel_labels))

    #Find which network each parcel belongs to
    for i in range(0,len(parcel_labels)):

        network_id_found = 0

        for j in range(0,len(network_identifiers)):

            if network_identifiers[j] in parcel_labels[i]:
                network_ids[i] = j
                network_id_found = 1
                all_networks_present[j] = 1

        if network_id_found == 0:

            for j in range(0,len(network_identifiers)):

                if alt_network_identifiers[j] in parcel_labels[i]:
                    network_ids[i] = j
                    network_id_found = 1
                    all_networks_present[j] = 1

        #Check that the parcel belongs to a network
        if network_id_found == 0:
            #pass
            if exclude_no_network_rois == False:
                raise NameError('Error - network not found for parcel: ' + parcel_labels[i])
            else:
                no_network_inds[i] = 1
                
    network_ids = network_ids[no_network_inds == 0]
    connectivity_matrix = connectivity_matrix[no_network_inds == 0,:]
    connectivity_matrix = connectivity_matrix[:,no_network_inds == 0]
    print('{} ROIs were unable to be mapped to a network'.format(np.sum(no_network_inds)))

    #Check the each network has a parcel
    if np.sum(all_networks_present) != all_networks_present.shape[0]:

        inds = np.where(all_networks_present == 0)
        network_string = ''
        for temp_ind in inds[0]:
            print(temp_ind)
            network_string = network_string + network_identifiers[temp_ind] + ' (Primary ID)/' + alt_network_identifiers[temp_ind] + ' (Possible Secondary ID), '

        raise NameError('Error - no parcels found for networks: ' + network_string[:-2])



    #Create arrays for the sorted network ids and also store the inds to
    #obtain the sorted matrix
    sorted_ids = np.sort(network_ids, kind = 'mergesort')
    sorted_id_inds = np.argsort(network_ids, kind = 'mergesort')

    #Calculate where the center and edge of each network is for labeling
    #different networks on netmat figures
    network_edges = np.zeros(len(network_names))
    for i in range(len(network_names)):
        for j in range(len(sorted_ids)):

            if sorted_ids[j] == i:

                network_edges[i] = j

    network_centers = np.zeros(len(network_names))
    network_centers[0] = network_edges[0]/2.0 - 0.5
    for i in range(1,len(network_edges)):
        network_centers[i] = (network_edges[i] + network_edges[i-1])/2.0


    #Sort the connectivity matrix to be aligned with networks
    sorted_conn_matrix = np.zeros(connectivity_matrix.shape)
    sorted_conn_matrix = np.reshape(connectivity_matrix[sorted_id_inds,:], connectivity_matrix.shape)
    sorted_conn_matrix = np.reshape(sorted_conn_matrix[:,sorted_id_inds], connectivity_matrix.shape)

    #Apply colormap to data
    cmap = getattr(matplotlib.cm, matplotlib_color_scheme)
    norm = matplotlib.colors.Normalize(vmin=minmax[0], vmax=minmax[1])
    m = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
    jet_conn_matrix =  m.to_rgba(sorted_conn_matrix, norm=True)

    #Add border to matrix
    jet_conn_with_borders = np.zeros((jet_conn_matrix.shape[0] + border_width, jet_conn_matrix.shape[1] + border_width, \
                                      jet_conn_matrix.shape[2]))
    jet_conn_with_borders[0:(-1*border_width),border_width:,:] = jet_conn_matrix
    for i in range(0,sorted_ids.shape[0]):
        jet_conn_with_borders[i,0:border_width,:] = network_colors[int(sorted_ids[i])]
        jet_conn_with_borders[(-1*border_width):,i+border_width,:] = network_colors[int(sorted_ids[i])]

    ######################################################################################
    ######################################################################################

    #Make and plot figure
    fig = plt.figure(dpi=dpi)
    plot_obj = plt.imshow(jet_conn_with_borders)
    if len(title) > 0:
        plt.title(title)

    lw = linewidth
    #Add lines to identify network borders
    for i in network_edges:
        plt.axvline(x=i + 0.5 + border_width,color=linecolor, lw=lw)
    plt.axvline(x=border_width - 0.5, color=linecolor, lw=lw)

    for i in network_edges:
        plt.axhline(y=i + 0.5,color=linecolor, lw=lw)

    ######################################################################################
    ######################################################################################

    #optionally add x tick labels
    if x_tick_labels:
        plt.xticks(ticks=network_centers + border_width + 0.5,labels=network_names, rotation=90, fontsize=x_tick_font_size)

    #optionally add y tick labels
    if y_tick_labels:
        plt.yticks(ticks=network_centers + 0.5,labels=network_names, fontsize=y_tick_font_size)

    #optionally add colorbar
    if add_colorbar:
        ax = plt.gca()
        mappable = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.15)
        plt.colorbar(mappable = mappable, cax = cax)


    return #fig


def make_network_bar_chart(regions, measure_values_by_region, xlabel = ''):
    #The names of the different networks for the visualization
    network_names = ['Vis. A', 'Vis. B', 'SomMot. A', 'SomMot. B', 'Temp. Par.', 'Dors. Attn. A',
                    'Dors. Attn. B', 'Sal. A', 'Sal. B', 'Cont. A', 'Cont. B', 'Cont. C', 'DMN A',
                    'DMN B', 'DMN C', 'Limbic A', 'Limbic B']

    #The name of different networks to pull out of the labels

    alt_network_identifiers = ['VisCent','VisPeri','SomMotA','SomMotB','TempPar', 'DorsAttnA',
                       'DorsAttnB','SalVentAttnA','SalVentAttnB','ContA','ContB','ContC','DefaultA',
                       'DefaultB','DefaultC','LimbicA','LimbicB']

    #The names of the different networks for the visualization
    network_names = ['Vis. A', 'Vis. B', 'SomMot. A', 'SomMot. B', 'Temp. Par.', 'Dors. Attn. A',
                    'Dors. Attn. B', 'Sal. A', 'Sal. B', 'Cont. A', 'Cont. B', 'Cont. C', 'DMN A',
                    'DMN B', 'DMN C', 'Limbic A', 'Limbic B']

    #The name of different networks to pull out of the labels

    alt_network_identifiers = ['VisCent','VisPeri','SomMotA','SomMotB','TempPar', 'DorsAttnA',
                       'DorsAttnB','SalVentAttnA','SalVentAttnB','ContA','ContB','ContC','DefaultA',
                       'DefaultB','DefaultC','LimbicA','LimbicB']

    subcortical_identifiers = ['AMY', 'HIP']
    subcortical_names = ['Amygdala', 'Hippocampus']

    subcortical_colors = [[0,0,0,1], #amygdala
                          [0.5,0.5,0.5,1]] #hippocampus

    network_colors = [
        [97/255, 38/255, 107/255, 1], #vis. a
        [195/255, 40/255, 39/255, 1], #vis. b
        [79/255, 130/255, 165/255, 1], #sommot a
        [82/255, 181/255, 140/255, 1], #sommat b
        [53/255, 75/255, 159/255, 1], #temp par
        [75/255, 147/255, 72/255, 1], #dors attn a
        [50/255, 116/255, 62/255, 1], #dors attn b
        [149/255, 77/255, 158/255, 1], #sal A
        [222/255, 130/255, 177/255, 1], #sal B
        [210/255, 135/255, 48/255, 1], #cont a
        [132/255, 48/255, 73/255, 1], #cont b
        [92/255, 107/255, 130/255, 1], #cont c
        [217/255, 221/255, 72/255, 1], #dmn a
        [176/255, 49/255, 69/255, 1], #dmn b
        [41/255, 37/255, 99/255, 1], #dmn c
        [75/255, 87/255, 61/255, 1], #limbic a
        [149/255, 166/255, 110/255, 1] #limbic b
    ]

    #Look at what regions/networks the Hariri ISC PCs map onto
    temp_dict = {}
    for a in alt_network_identifiers:
        temp_dict[a] = []
    temp_dict['AMY'] = []
    temp_dict['HIP'] = []
    temp_dict['Other'] = []
    bar_chart_names = ['Vis. A', 'Vis. B', 'SomMot. A', 'SomMot. B', 'Temp. Par.', 'Dors. Attn. A',
                    'Dors. Attn. B', 'Sal. A', 'Sal. B', 'Cont. A', 'Cont. B', 'Cont. C', 'DMN A',
                    'DMN B', 'DMN C', 'Limbic A', 'Limbic B', 'Amyg.', 'Hipp.', 'Other Subcort.']
    bar_chart_entries = ['VisCent','VisPeri','SomMotA','SomMotB','TempPar', 'DorsAttnA',
                       'DorsAttnB','SalVentAttnA','SalVentAttnB','ContA','ContB','ContC','DefaultA',
                       'DefaultB','DefaultC','LimbicA','LimbicB', 'AMY', 'HIP', 'Other']
    bar_chart_colors = [
        [97/255, 38/255, 107/255, 1], #vis. a
        [195/255, 40/255, 39/255, 1], #vis. b
        [79/255, 130/255, 165/255, 1], #sommot a
        [82/255, 181/255, 140/255, 1], #sommat b
        [53/255, 75/255, 159/255, 1], #temp par
        [75/255, 147/255, 72/255, 1], #dors attn a
        [50/255, 116/255, 62/255, 1], #dors attn b
        [149/255, 77/255, 158/255, 1], #sal A
        [222/255, 130/255, 177/255, 1], #sal B
        [210/255, 135/255, 48/255, 1], #cont a
        [132/255, 48/255, 73/255, 1], #cont b
        [92/255, 107/255, 130/255, 1], #cont c
        [217/255, 221/255, 72/255, 1], #dmn a
        [176/255, 49/255, 69/255, 1], #dmn b
        [41/255, 37/255, 99/255, 1], #dmn c
        [75/255, 87/255, 61/255, 1], #limbic a
        [149/255, 166/255, 110/255, 1], #limbic b
        [0.3, 0.3, 0.3, 1], #amy
        [0.7, 0.7, 0.7, 1], #hip
        [0, 0, 0, 1], #other
    ]

    temp_dict = {}
    for a in bar_chart_entries:
        temp_dict[a] = []

    for r, temp_region in enumerate(regions):
        found = 0
        try:
            if (temp_region.split('_')[2] in alt_network_identifiers) == True:
                for i in range(len(alt_network_identifiers)):
                    if (temp_region.split('_')[2] in alt_network_identifiers[i]):
                        temp_dict[alt_network_identifiers[i]].append(measure_values_by_region[r])
                        found = 1
        except:
            if 'AMY' in temp_region:
                temp_dict['AMY'].append(measure_values_by_region[r])
            elif 'HIP' in temp_region:
                temp_dict['HIP'].append(measure_values_by_region[r])
            else:
                temp_dict['Other'].append(measure_values_by_region[r])            

    means = []
    std_devs = []
    for i in bar_chart_entries:
        means.append(np.mean(temp_dict[i]))
        std_devs.append(np.std(temp_dict[i]))
    plt.figure(dpi=150)
    plt.barh(bar_chart_names, means, color=bar_chart_colors, xerr=std_devs)
    plt.xlabel(xlabel)
    
    return


import statsmodels.api as sm
from patsy import dmatrices


def fast_regression_beta_resids(Y, X_PINV, X):

    '''Function that calculates regression

    Based on the pseudo-inverse, calculates
    beta-weights and residuals for linear
    regression problem to predict values
    of Y

    Parameters
    ----------

    Y : numpy.ndarray
        Dependent variables to be predicted.
        Shape <n> or <n,1>
    X_PINV : numpy.ndarray
        The pseudo-inverse for independent variables
        X used as predictors in linear regression.
        This is the result of np.linalg.pinv(X)
    X : numpy.ndarray

    Returns
    -------

    B : numpy.ndarray
        The beta-weights for the linear regression
    residuals : numpy.ndarray
        The residuals of the least squares model
    '''


    B = np.matmul(X_PINV, Y)
    residuals = Y - np.matmul(X, B)
    return B, residuals

def calc_model_comparison(full_X, partial_X, Y):
    
    '''
    Parameters
    ----------
    full_X : numpy.ndarray (n_samples, n_predictors)
        All regressors that will have coefficients fit
    partial_X : numpy.ndarray (n_samples, n_partial_predictors)
        Same as above except for n_partial_predictors includes
        predictors that are a subset of n_predictors
    Y : numpy.ndarray (n_samples,)
    
    Returns
    -------
    P-Value, F-Statistic, F-Dist DOF 1, F-Dist DOF 2
    
    '''
    
    full_B = np.matmul(np.linalg.pinv(full_X), Y)
    full_residuals = Y - np.matmul(full_X, full_B)

    partial_B = np.matmul(np.linalg.pinv(partial_X), Y)
    partial_residuals = Y - np.matmul(partial_X, partial_B)

    ss_full = np.sum(full_residuals**2)
    ss_partial = np.sum(partial_residuals**2)

    df_full = Y.shape[0] - full_X.shape[1]
    F = (ss_partial - ss_full)/(full_X.shape[1] - partial_X.shape[1])/(ss_full/df_full)
    p = 1-scipy.stats.f.cdf(F, full_X.shape[1] - partial_X.shape[1], df_full)
    
    return p, F, full_X.shape[1] - partial_X.shape[1], df_full, full_B, partial_B

def fit_reduced_model_for_fl(phenotype_data, reduced_dmat_x):
    '''Calculate betas and residuals for freedman-lane
    
    This function takes some multidimensional img_data
    with at least one feature, and mutliple samples, 
    and regresses out the effects found in the reduced_dmat_x
    design matrix. The residuals and beta weights generated by
    this are then return so that Freedman-Lane type permutations
    can be done
    
    
    Parameters
    ----------
    
    img_data : numpy.ndarray
        Data array with shape <n_samples, n_features>
    reduced_dmat_x : numpy.ndarray
        This is a design matrix that will be used to
        fit a glm to the different features of img_data.
        This design matrix should NOT contain the variable
        you are interested in conducting inference on,
        so these variables are called "nuisance" variables.
    
    Returns
    -------
    
    reduced_residuals : numpy.ndarray
        Array containing residuals with shape <n_samples, n_img_features>
    reduced_betas : numpy.ndarray
        Calculated beta weights with shape <n_img_features, n_dmat_features>
    '''
       
    reduced_pinv = np.linalg.pinv(reduced_dmat_x)
    reduced_betas = np.zeros((phenotype_data.shape[1], reduced_dmat_x.shape[1]))
    reduced_residuals = np.zeros(phenotype_data.shape)
    for i in range(phenotype_data.shape[1]):

        temp_betas, temp_residuals = fast_regression_beta_resids(phenotype_data[:,i], reduced_pinv, reduced_dmat_x)
        reduced_betas[i,:] = temp_betas
        reduced_residuals[:,i] = temp_residuals
        
    return reduced_residuals, reduced_betas

def generate_fl_permuted_data(img_data_shape, reduced_residuals, reduced_dmat_x, reduced_betas):
    
    '''Generates permuted data based on Freedman-Lane

    The purpose of this function is to create a permuted
    version of a set of dependent variables (such as a
    set of neuroimaging features) that can be used for
    valid permutations in the presence of nuisance variables.
    
    Prior to calling this function, a GLM should be fit to
    your dependent variables using the nuisance variables that
    you aren't interested in making statistical inference on.
    This procedure is how you come up with reduced_residuals,
    reduced_dmat_x, and reduced_betas 
    
    Based on these input features, a new set up permuted data
    will be generated.
    
    See Anderson Winkler's blog for walkthrough of freedman-lane
    procedure:
    https://brainder.org/2020/05/19/simplifying-freedman-lane/
    
    Freedman-Lane citation:
    David Freedman and David Lane. A nonstochastic interpretation of
    reported significance levels, Journal of Business & Economic
    Statistics, 1983


    Parameters
    ----------

    img_data_shape : tuple
        Shape of your image data (i.e. dependent variables)
    reduced_residuals : numpy.ndarray
    reduced_dmat_x : numpy.ndarray
    reduced_betas : numpy.ndarray
        
    Returns
    -------

    permuted_img_data : numpy.ndarray
        Permuted image data
    '''
    
    
    #Create permuted image data (with residuals from reduced model)
    permuted_img_data = np.zeros(img_data_shape)
    temp_perms = np.random.permutation(img_data_shape[0]).astype(int)
    for j in range(img_data_shape[1]):
        permuted_img_data[:,j] = reduced_residuals[temp_perms,j] + np.matmul(reduced_dmat_x, reduced_betas[j,:])
        
    return permuted_img_data

def make_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms):

    behav_df = pd.read_csv(csv_path, index_col = 0)
    behav_phenotype_name = phenotype_name
    behav_phenotype_name_unedited = behav_phenotype_name
    log_transform_behav_phenotype = log_transform
    behav_phenotypes = []
    ages = []
    pcs = []
    ics = []
    iscs_for_phenotype_prediction = []
    phenotype_subj_paths = []
    phenotype_sessions = []
    timeseries_for_phenotype_subjects = []
    for i, temp_subject in enumerate(isc_subjects):

        int_session = int(isc_sessions[i].split('-')[-1])
        subset_df = behav_df[behav_df['session_id'] == int_session]
        if subset_df.shape[0] == 1:
            if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
                pcs.append(hariri_pc_scores[i,:num_pcs])
                ics.append(all_hariri_isc_mats[i,:num_pcs])
                ages.append(subset_df['age'].values[0])
                behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
                iscs_for_phenotype_prediction.append(hariri_isc_mats[i, :])
                phenotype_subj_paths.append(temp_subject)
                phenotype_sessions.append(isc_sessions[i])

                found = 0
                for temp_key in all_runs_timeseries_dict.keys():
                    if (temp_subject in temp_key) and (isc_sessions[i] in temp_key):
                        found += 1
                        timeseries_for_phenotype_subjects.append(all_runs_timeseries_dict[temp_key])
                if found != 1:
                    raise NameError('Error: something is wrong with code')

    iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
    pcs = np.array(pcs)
    ics = np.array(ics)
    constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
    ages = np.array(ages)[:,None]

    if log_transform_behav_phenotype:
        behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
        behav_phenotype_name = 'log_' + behav_phenotype_name
    else:
        behav_phenotypes = np.array(behav_phenotypes)[:,None]

    df = {}
    pcs_text = ''
    ics_text = ''
    for i in range(num_pcs):
        df['PC_{}'.format(i)] = pcs[:,i]
        df['IC_{}'.format(i)] = ics[:,i]
        pcs_text = pcs_text + 'PC_{} + '.format(i)
        ics_text = ics_text + 'IC_{} + '.format(i)

    df[behav_phenotype_name] = behav_phenotypes
    df['age'] = ages


    Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
    Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
    model_full = sm.OLS(Y,X_Full)
    results_full = model_full.fit()
    
    pc_model_loading = np.zeros(hariri_isc_pca.components_.shape[1])
    for temp_pc in range(num_pcs):
        pc_text = 'PC_{}'.format(temp_pc) ###THIS USED TO BE WRONG AND HAD BEEN SET AS "i"
        pc_model_loading += hariri_isc_pca.components_[temp_pc, :]*results_full.params[pc_text]
    make_network_bar_chart(regions, pc_model_loading, xlabel = 'PC Model Loading')
            
    
    full_predictions = results_full.predict(X_Full)
    full_predictions_minus_age_effect = full_predictions - X_Full['age']*results_full.params['age']
    #Y_minus_age_effect = Y.values.squeeze() - X_Full['age'].values.squeeze()*results_full.params['age'].squeeze()
    #print(Y_minus_age_effect)
    print(full_predictions_minus_age_effect)
    print(full_predictions_minus_age_effect.shape)
    plt.figure(dpi=200)
    plt.scatter(full_predictions_minus_age_effect, Y.values)
    plt.xlabel('Predicted {} - Predicted Age Effect'.format(behav_phenotype_name))
    plt.ylabel('Actual {}'.format(behav_phenotype_name))
    print(np.corrcoef(full_predictions_minus_age_effect, Y.values.squeeze()))
    
    haufe_transform_pc_corr_coefs = np.zeros(hariri_isc_mats.shape[1])
    for temp_region in range(hariri_isc_mats.shape[1]):
        haufe_transform_pc_corr_coefs[temp_region] = np.corrcoef(iscs_for_phenotype_prediction[:,temp_region].squeeze(), full_predictions)[0,1]
    make_network_bar_chart(regions, haufe_transform_pc_corr_coefs, xlabel = 'Haufe-Transformed Effect Size (Corr)')
    cifti_output_name = 'hariri_isc_with_{}_HaufeTransformedPCModelEffects_CorrCoef.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(haufe_transform_pc_corr_coefs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    haufe_transform_pc_covs = np.zeros(hariri_isc_mats.shape[1])
    for temp_region in range(hariri_isc_mats.shape[1]):
        haufe_transform_pc_covs[temp_region] = np.cov(iscs_for_phenotype_prediction[:,temp_region].squeeze(), full_predictions)[0,1]
    make_network_bar_chart(regions, haufe_transform_pc_covs, xlabel = 'Haufe-Transformed Effect Size (Cov)')
    cifti_output_name = 'hariri_isc_with_{}_HaufeTransformedPCModelEffects_Covariances.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(haufe_transform_pc_covs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    
    

    results_partial = sm.OLS(Y, X_Partial).fit()
    print('Test for {} comparing full/reduced model: f = {}, p = {}, df_diff = {}'.format(behav_phenotype_name, results_full.compare_f_test(results_partial)[0], results_full.compare_f_test(results_partial)[1], results_full.compare_f_test(results_partial)[2]))
    baseline_predictions = results_full.predict(X_Full)
    print('Full model results summary: \n\n')
    print(results_full.summary())
    
    print('\n\nIndividual Associations (corr coef) between ISC-PCs and {} (first session only, n = {})'.format(behav_phenotype_name, df[behav_phenotype_name].squeeze().shape))
    for i in range(num_pcs):
        print('PC_{} : {}'.format(i, np.corrcoef(df['PC_{}'.format(i)], df[behav_phenotype_name].squeeze())[0,1]))
        
        
    orig_behav_phenotypes = []
    orig_ages = []
    orig_pcs = []
    orig_iscs_for_phenotype_prediction = []



    follow_up_behav_phenotypes = []
    follow_up_ages = []
    follow_up_pcs = []
    follow_up_iscs_for_phenotype_prediction = []

    for i, temp_subject in enumerate(isc_subjects):


        temp_subject = temp_subject.split('/')[-1]
        temp_session_inds = np.where(np.array(all_isc_subjects) == temp_subject)
        temp_sessions = np.array(all_isc_sessions)[temp_session_inds]
        new_sessions = []
        for temp_ses in temp_sessions:
            if isc_sessions[i] in temp_ses:
                continue
            else:
                new_sessions.append(temp_ses)
        if len(new_sessions) == 0:
            continue
        new_sessions.sort()
        next_session = new_sessions[0]
        int_session = int(next_session.split('-')[-1])
        subset_df = behav_df[behav_df['session_id'] == int_session]

        first_int_session = int(isc_sessions[i].split('-')[-1])
        first_session_df = behav_df[behav_df['session_id'] == first_int_session]

        if subset_df.shape[0]*first_session_df.shape[0]:
            first_req = (np.isnan(subset_df[behav_phenotype_name_unedited].values[0]) == False) and ((subset_df[behav_phenotype_name_unedited].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False)
            secon_req = (np.isnan(first_session_df[behav_phenotype_name_unedited].values[0]) == False) and ((first_session_df[behav_phenotype_name_unedited].values[0]) >= 0) and (np.isnan(first_session_df['age'].values[0]) == False)
            if first_req and secon_req:
                follow_up_pcs.append(all_hariri_pc_scores[np.where(next_session == np.array(all_isc_sessions))[0][0],:num_pcs])
                follow_up_ages.append(subset_df['age'].values[0])
                follow_up_behav_phenotypes.append(subset_df[behav_phenotype_name_unedited].values[0])
                follow_up_iscs_for_phenotype_prediction.append(all_hariri_isc_mats[np.where(next_session == np.array(all_isc_sessions))[0][0], :])

                ###### Now get the original session data for these subjects
                orig_pcs.append(hariri_pc_scores[i,:num_pcs])
                orig_ages.append(first_session_df['age'].values[0])
                orig_behav_phenotypes.append(first_session_df[behav_phenotype_name_unedited].values[0])
                orig_iscs_for_phenotype_prediction.append(hariri_isc_mats[i, :])


    follow_up_iscs_for_phenotype_prediction = np.array(follow_up_iscs_for_phenotype_prediction)
    follow_up_pcs = np.array(follow_up_pcs)
    follow_up_constant = np.ones(follow_up_iscs_for_phenotype_prediction.shape[0])[:,None]
    follow_up_ages = np.array(follow_up_ages).squeeze()[:,None]

    orig_iscs_for_phenotype_prediction = np.array(orig_iscs_for_phenotype_prediction)
    orig_pcs = np.array(orig_pcs)
    orig_constant = np.ones(follow_up_iscs_for_phenotype_prediction.shape[0])[:,None]
    orig_ages = np.array(orig_ages).squeeze()[:,None]

    if log_transform_behav_phenotype:
        follow_up_behav_phenotypes = np.log(np.array(follow_up_behav_phenotypes) + 1)[:,None]
        orig_behav_phenotypes = np.log(np.array(orig_behav_phenotypes) + 1)[:,None]
    else:
        follow_up_behav_phenotypes = np.array(follow_up_behav_phenotypes)[:,None]
        orig_behav_phenotypes = np.array(orig_behav_phenotypes)[:,None]
        
        
    follow_up_df = {}
    orig_df = {}
    pcs_text = ''
    for i in range(num_pcs):
        follow_up_df['PC_{}'.format(i)] = follow_up_pcs[:,i]
        orig_df['PC_{}'.format(i)] = orig_pcs[:,i]
        pcs_text = pcs_text + 'PC_{} + '.format(i)

    follow_up_df[behav_phenotype_name] = follow_up_behav_phenotypes
    orig_df[behav_phenotype_name] = orig_behav_phenotypes
    follow_up_df['age'] = follow_up_ages
    orig_df['age'] = orig_ages
    follow_up_Y, follow_up_X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, pcs_text), data=follow_up_df, return_type='dataframe')
    follow_up_Y, follow_up_X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=follow_up_df, return_type='dataframe')
    orig_Y, orig_X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, pcs_text), data=orig_df, return_type='dataframe')
    new_follow_up_model = sm.OLS(follow_up_Y, follow_up_X_Full)
    new_follow_up_model_partial = sm.OLS(follow_up_Y, follow_up_X_Partial)
    new_follow_up_model_results = new_follow_up_model.fit()
    new_follow_up_model_results_partial = new_follow_up_model_partial.fit()
    print('\n\nLinear Regression Trained on Follow-Up Data F-Test: {}'.format(new_follow_up_model_results.compare_f_test(new_follow_up_model_results_partial)))
    print(new_follow_up_model_results.summary())

    ######################################################################
    ######################################################################
    ######################################################################
    #Now ROI based approach
    isc_pvals = np.zeros(hariri_isc_mats.shape[1])
    isc_tvals = np.zeros(hariri_isc_mats.shape[1])
    isc_coefs = np.zeros(hariri_isc_mats.shape[1])
    isc_slopes = np.zeros(hariri_isc_mats.shape[1])

    for i in range(isc_pvals.shape[0]):
        X_Full = np.hstack([iscs_for_phenotype_prediction[:,i].squeeze()[:,None], ages, constant])
        X_Partial = np.hstack([ages, constant])
        Y = behav_phenotypes
        
        #p, f, d1, d2, betas_full, betas_partial = calc_model_comparison(X_Full, X_Partial, Y)
        
        model_full = sm.OLS(Y,X_Full)
        results_full = model_full.fit()

        results_partial = sm.OLS(Y, X_Partial).fit()
        results_full.compare_f_test(results_partial)
        isc_pvals[i] = -1*np.log10(results_full.compare_f_test(results_partial)[1])
        #isc_pvals[i] = -1*np.log10(p)
        #isc_coefs[i] = betas_full[0]
        isc_tvals[i] = results_full._results.tvalues[0]
        isc_slopes[i] = results_full.params[0]
        
        if i == 186:
            print(results_full.summary())

    cifti_output_name = 'hariri_isc_with_{}_neglog10p_uncorrected.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(isc_pvals[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    #cifti_output_name = 'hariri_isc_with_{}_coefs.pscalar.nii'.format(behav_phenotype_name)
    #make_pscalar_cifti(isc_coefs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

    cifti_output_name = 'hariri_isc_with_{}_tvalues_uncorrected.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(isc_tvals[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    
    plt.figure(dpi = 150)
    plt.hist(isc_pvals, bins = 20)
    plt.title('Distribution of Second-Level Stats Across Regions ({})'.format(behav_phenotype_name))
    plt.ylabel('Count')
    plt.xlabel('-log10(p-value)')
    
    make_network_bar_chart(regions, isc_tvals, xlabel = 'test-statistic (uncorrected)')
    
    ########################################################################################
    ########################################################################################
    ########################################################################################
    img_data = np.array(iscs_for_phenotype_prediction)
    reduced_residuals, reduced_betas = fit_reduced_model_for_fl(behav_phenotypes, X_Partial)

    null_dist = np.zeros(num_perms)
    null_isc_pvals = np.zeros(hariri_isc_mats.shape[1])

    for i in range(num_perms):

        #Generate permuted phenotype data (y) using freedman lane procedure
        permuted_phenotype_data = generate_fl_permuted_data(behav_phenotypes.shape, reduced_residuals, X_Partial, reduced_betas)
        if np.mod(i, 200) == 0:
            print(i)

        #Fit modle for each ROI
        for j in range(null_isc_pvals.shape[0]):
            X_Full = np.hstack([iscs_for_phenotype_prediction[:,j][:,None], ages, constant])
            X_Partial = np.hstack([ages, constant])
            Y = permuted_phenotype_data
            
            #p, f, d1, d2, betas_full, betas_partial = calc_model_comparison(X_Full, X_Partial, Y)
            
            model_full = sm.OLS(Y,X_Full)
            results_full = model_full.fit()

            results_partial = sm.OLS(Y, X_Partial).fit()
            null_isc_pvals[j] = -1*np.log10(results_full.compare_f_test(results_partial)[1])
            #results_full.compare_f_test(results_partial)
            #null_isc_pvals[j] = -1*np.log10(p)

        #Save the best statistic (neglog10(p)) found across ROIs for the current permutation
        null_dist[i] = np.max(null_isc_pvals)
        
    ###########################################################################################
    ###########################################################################################
    
    alpha_5th_percentile_cutoff = np.percentile(null_dist, 95)
    print("5th percentile cutoff (-10log(p)): {}".format(alpha_5th_percentile_cutoff))
    surviving_inds = np.where(isc_pvals > alpha_5th_percentile_cutoff)[0]
    print("Regions surviving corrections at alpha = 0.05: {}".format(regions[surviving_inds]))
    print("Inds surviving correction at alpha = 0.05: {}".format(surviving_inds))
    pvals = []
    for temp_ind in surviving_inds:
        pvals.append(np.sum(null_dist > isc_pvals[temp_ind])/num_perms)
    print("Corrected P-Vals of Inds surviving correction at alpha = 0.05: {}".format(pvals))

    ###Update this to work for arbritrary ROIs.....
    plt.figure(dpi = 150)
    plt.hist(null_dist, bins = 200)
    plt.title('Null Distribution')
    plt.ylabel('Count')
    plt.xlabel('-10log(p-value)')
    plt.axvline(isc_pvals[186], linestyle = '--', color = 'red')
    plt.legend(['17Networks_LH_DefaultA_pCunPCC_2', 'Null Distribution'])
    
    return isc_tvals, isc_slopes
    
    
def calc_weighted_corr_coef(signal_1, signal_2, weighting):
    
    '''Function to calculate a weighted correlation coefficient
    
    Parameters
    ----------
    signal_1 : numpy.ndarray <n,>
    signal_2 : numpy.ndarray <n,>
    weighting: numpy.ndarray <n,>
    
    Returns
    -------
    Correlation coefficient calculated based on weighted means/vars/covs
    using the absolute value of the weighting variable to determine the relative
    contribution
    
    '''
    
    absolute_weighting = np.absolute(weighting)
    sum_normalized_weighting_arr = absolute_weighting/np.sum(absolute_weighting)
    weighted_mean_1 = np.sum(np.multiply(sum_normalized_weighting_arr, signal_1))
    weighted_mean_2 = np.sum(np.multiply(sum_normalized_weighting_arr, signal_2))
    weighted_cov = np.sum((signal_1 - weighted_mean_1)*(signal_2 - weighted_mean_2)*sum_normalized_weighting_arr)
    weighted_std_1 = np.sqrt(np.sum(((signal_1 - weighted_mean_1)**2)*sum_normalized_weighting_arr))
    weighted_std_2 = np.sqrt(np.sum(((signal_2 - weighted_mean_2)**2)*sum_normalized_weighting_arr))
    weighted_corr = weighted_cov/(weighted_std_1*weighted_std_2)
    
    return weighted_corr


def apply_filter(b, a, signal):

    """
    #Wrapper function to apply the filter coefficients from
    #construct_filter to a signal.
    #should have filtfilt imported from scipy.signal
    """

    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal


def construct_filter(btype, cutoff, TR, order):

    """
    #btype should be 'lowpass', 'highpass', or 'bandpass' and
    #cutoff should be list (in Hz) with length 1 for low and high and
    #2 for band. Order is the order of the filter
    #which will be doubled since filtfilt will be used
    #to remove phase distortion from the filter. Recommended
    #order is 6. Will return filter coefficients b and a for
    #the desired butterworth filter.
    #Constructs filter coefficients. Use apply_filter to use
    #the coefficients to filter a signal.
    #Should have butter imported from scipy.signal
    """


    nyq = 0.5 * (1/TR)

    if btype == 'lowpass':
        if len(cutoff) != 1:
            raise NameError('Error: lowpass type filter should have one cutoff values')
        low = cutoff[0]/nyq
        b, a = butter(order, low, btype='lowpass')

    elif btype == 'highpass':
        if len(cutoff) != 1:
            raise NameError('Error: highpass type filter should have one cutoff values')
        high = cutoff[0]/nyq
        b, a = butter(order, high, btype='highpass')

    elif btype == 'bandpass':
        if len(cutoff) != 2:
            raise NameError('Error: bandpass type filter should have two cutoff values')
        low = min(cutoff)/nyq
        high = max(cutoff)/nyq
        b, a = butter(order, [low, high], btype='bandpass')

    else:
        raise NameError('Error: filter type should by low, high, or band')


    return b, a


def make_behav_predictions_task(cross_sectional_task_pcs, cross_sectional_task_data, csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms):

    behav_df = pd.read_csv(csv_path, index_col = 0)
    behav_phenotype_name = phenotype_name
    behav_phenotype_name_unedited = behav_phenotype_name
    log_transform_behav_phenotype = log_transform
    behav_phenotypes = []
    ages = []
    task_pcs = []
    task_contrasts_for_phenotype_prediction = []
    phenotype_subj_paths = []
    phenotype_sessions = []
    timeseries_for_phenotype_subjects = []
    task_activations = []
    for i, temp_subject in enumerate(isc_subjects):

        int_session = int(isc_sessions[i].split('-')[-1])
        subset_df = behav_df[behav_df['session_id'] == int_session]
        if subset_df.shape[0] == 1:
            if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
                task_pcs.append(cross_sectional_task_pcs[i,:num_pcs])
                ages.append(subset_df['age'].values[0])
                behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
                task_contrasts_for_phenotype_prediction.append(cross_sectional_task_data[i, :])
                phenotype_subj_paths.append(temp_subject)
                phenotype_sessions.append(isc_sessions[i])

                found = 0
                for temp_key in all_runs_timeseries_dict.keys():
                    if (temp_subject in temp_key) and (isc_sessions[i] in temp_key):
                        found += 1
                        timeseries_for_phenotype_subjects.append(all_runs_timeseries_dict[temp_key])
                if found != 1:
                    raise NameError('Error: something is wrong with code')

    task_contrasts_for_phenotype_prediction = np.array(task_contrasts_for_phenotype_prediction)
    task_pcs = np.array(task_pcs)
    constant = np.ones(task_contrasts_for_phenotype_prediction.shape[0])[:,None]
    ages = np.array(ages)[:,None]

    if log_transform_behav_phenotype:
        behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
        behav_phenotype_name = 'log_' + behav_phenotype_name
    else:
        behav_phenotypes = np.array(behav_phenotypes)[:,None]

    df = {}
    pcs_text = ''
    for i in range(num_pcs):
        df['PC_{}'.format(i)] = task_pcs[:,i]
        pcs_text = pcs_text + 'PC_{} + '.format(i)

    df[behav_phenotype_name] = behav_phenotypes
    df['age'] = ages


    Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
    Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
    model_full = sm.OLS(Y,X_Full)
    results_full = model_full.fit()

    results_partial = sm.OLS(Y, X_Partial).fit()
    print('Test for {} comparing full/reduced model: f = {}, p = {}'.format(behav_phenotype_name, results_full.compare_f_test(results_partial)[0], results_full.compare_f_test(results_partial)[1]))
    baseline_predictions = results_full.predict(X_Full)
    print('Full model results summary: \n\n')
    print(results_full.summary())
    
    
    full_predictions = results_full.predict(X_Full)
    full_predictions_minus_age_effect = full_predictions - X_Full['age']*results_full.params['age']
    plt.figure(dpi=200)
    plt.scatter(full_predictions_minus_age_effect, Y.values)
    plt.xlabel('Predicted {} - Predicted Age Effect'.format(behav_phenotype_name))
    plt.ylabel('Actual {}'.format(behav_phenotype_name))
    print(np.corrcoef(full_predictions_minus_age_effect, Y.values.squeeze()))
    
    haufe_transform_pc_corr_coefs = np.zeros(hariri_isc_mats.shape[1])
    for temp_region in range(hariri_isc_mats.shape[1]):
        haufe_transform_pc_corr_coefs[temp_region] = np.corrcoef(task_contrasts_for_phenotype_prediction[:,temp_region].squeeze(), full_predictions)[0,1]
    make_network_bar_chart(regions, haufe_transform_pc_corr_coefs, xlabel = 'Haufe-Transformed Effect Size (Corr)')
    cifti_output_name = 'hariri_EmotionMinusShape_with_{}_HaufeTransformedPCModelEffects_CorrCoef.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(haufe_transform_pc_corr_coefs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    haufe_transform_pc_covs = np.zeros(hariri_isc_mats.shape[1])
    for temp_region in range(hariri_isc_mats.shape[1]):
        haufe_transform_pc_covs[temp_region] = np.cov(task_contrasts_for_phenotype_prediction[:,temp_region].squeeze(), full_predictions)[0,1]
    make_network_bar_chart(regions, haufe_transform_pc_covs, xlabel = 'Haufe-Transformed Effect Size (Cov)')
    cifti_output_name = 'hariri_EmotionMinusShape_with_{}_HaufeTransformedPCModelEffects_Covariances.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(haufe_transform_pc_covs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    
    
    
    print('\n\nIndividual Associations (corr coef) between Task-PCs and {} (first session only, n = {})'.format(behav_phenotype_name, df[behav_phenotype_name].squeeze().shape))
    for i in range(num_pcs):
        print('PC_{} : {}'.format(i, np.corrcoef(df['PC_{}'.format(i)], df[behav_phenotype_name].squeeze())[0,1]))
        
    ######################################################################
    ######################################################################
    ######################################################################
    #Now ROI based approach
    task_pvals = np.zeros(task_contrasts_for_phenotype_prediction.shape[1])
    task_tvals = np.zeros(task_contrasts_for_phenotype_prediction.shape[1])
    task_coefs = np.zeros(task_contrasts_for_phenotype_prediction.shape[1])

    for i in range(task_pvals.shape[0]):
        X_Full = np.hstack([task_contrasts_for_phenotype_prediction[:,i].squeeze()[:,None], ages, constant])
        X_Partial = np.hstack([ages, constant])
        Y = behav_phenotypes
        
        #p, f, d1, d2, betas_full, betas_partial = calc_model_comparison(X_Full, X_Partial, Y)
        
        model_full = sm.OLS(Y,X_Full)
        results_full = model_full.fit()

        results_partial = sm.OLS(Y, X_Partial).fit()
        results_full.compare_f_test(results_partial)
        task_pvals[i] = -1*np.log10(results_full.compare_f_test(results_partial)[1])
        #isc_pvals[i] = -1*np.log10(p)
        #isc_coefs[i] = betas_full[0]
        task_tvals[i] = results_full._results.tvalues[0]

    cifti_output_name = 'hariri_EmotionGreaterThanShape_with_{}_neglog10p_uncorrected.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(task_pvals[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    #cifti_output_name = 'hariri_isc_with_{}_coefs.pscalar.nii'.format(behav_phenotype_name)
    #make_pscalar_cifti(isc_coefs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

    cifti_output_name = 'hariri_EmotionGreaterThanShape_with_{}_tvalues_uncorrected.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(task_tvals[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    
    plt.figure(dpi = 150)
    plt.hist(task_pvals, bins = 20)
    plt.title('Distribution of Second-Level Stats Across Regions ({})'.format(behav_phenotype_name))
    plt.ylabel('Count')
    plt.xlabel('-log10(p-value)')
    
    ########################################################################################
    ########################################################################################
    ########################################################################################
    img_data = np.array(task_contrasts_for_phenotype_prediction)
    reduced_residuals, reduced_betas = fit_reduced_model_for_fl(behav_phenotypes, X_Partial)

    null_dist = np.zeros(num_perms)
    null_task_pvals = np.zeros(task_contrasts_for_phenotype_prediction.shape[1])

    for i in range(num_perms):

        #Generate permuted phenotype data (y) using freedman lane procedure
        permuted_phenotype_data = generate_fl_permuted_data(behav_phenotypes.shape, reduced_residuals, X_Partial, reduced_betas)
        if np.mod(i, 200) == 0:
            print(i)

        #Fit modle for each ROI
        for j in range(null_task_pvals.shape[0]):
            X_Full = np.hstack([task_contrasts_for_phenotype_prediction[:,j][:,None], ages, constant])
            X_Partial = np.hstack([ages, constant])
            Y = permuted_phenotype_data
            
            #p, f, d1, d2, betas_full, betas_partial = calc_model_comparison(X_Full, X_Partial, Y)
            
            model_full = sm.OLS(Y,X_Full)
            results_full = model_full.fit()

            results_partial = sm.OLS(Y, X_Partial).fit()
            null_task_pvals[j] = -1*np.log10(results_full.compare_f_test(results_partial)[1])
            #results_full.compare_f_test(results_partial)
            #null_isc_pvals[j] = -1*np.log10(p)

        #Save the best statistic (neglog10(p)) found across ROIs for the current permutation
        null_dist[i] = np.max(null_task_pvals)
        
    ###########################################################################################
    ###########################################################################################
    
    alpha_5th_percentile_cutoff = np.percentile(null_dist, 95)
    print("5th percentile cutoff (-10log(p)): {}".format(alpha_5th_percentile_cutoff))
    surviving_inds = np.where(task_pvals > alpha_5th_percentile_cutoff)[0]
    print("Regions surviving corrections at alpha = 0.05: {}".format(regions[surviving_inds]))
    print("Inds surviving correction at alpha = 0.05: {}".format(surviving_inds))
    pvals = []
    for temp_ind in surviving_inds:
        pvals.append(np.sum(null_dist > task_pvals[temp_ind])/num_perms)
    print("Corrected P-Vals of Inds surviving correction at alpha = 0.05: {}".format(pvals))

    ###Update this to work for arbritrary ROIs.....
    plt.figure(dpi = 150)
    plt.hist(null_dist, bins = 200)
    plt.title('Null Distribution')
    plt.ylabel('Count')
    plt.xlabel('-10log(p-value)')
    plt.axvline(task_pvals[186], linestyle = '--', color = 'red')
    plt.legend(['17Networks_LH_DefaultA_pCunPCC_2', 'Null Distribution'])
    
    
    
    
def make_mixed_effects_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms, output_folder = None):

    isnone = type(output_folder) == type(None)
    if (isnone == False):
        if os.path.exists(output_folder) == False:
            os.makedirs(output_folder)
    
    ########################
    #Not sure if we need to use these three yet
    phenotype_subj_paths = []
    phenotype_sessions = []
    timeseries_for_phenotype_subjects = []
    ##########################
    
    
    ########################################
    ############THis is from Mixed Effects##
    behav_df = pd.read_csv(csv_path, index_col = 0)
    behav_phenotype_name = phenotype_name
    behav_phenotype_name_unedited = behav_phenotype_name
    log_transform_behav_phenotype = log_transform
    behav_phenotypes = []
    ages = []
    pcs = []
    visits = [] #This is not included in GLM results
    iscs_for_phenotype_prediction = []
    subjects_for_long_prediction = []

    for i, temp_subject in enumerate(all_isc_subjects):

        int_session = int(all_isc_sessions[i].split('-')[-1])
        subset_df = behav_df[behav_df['session_id'] == int_session]
        if subset_df.shape[0] == 1:
            if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
                pcs.append(all_hariri_pc_scores[i,:num_pcs])
                ages.append(subset_df['age'].values[0])
                behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
                iscs_for_phenotype_prediction.append(all_hariri_isc_mats[i, :])
                subjects_for_long_prediction.append(temp_subject)
                visits.append(int(subset_df.index.values[0].split('_')[1]))

    iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
    pcs = np.array(pcs)
    constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
    ages = np.array(ages)[:,None]
    visits = np.array(visits)

    if log_transform_behav_phenotype:
        behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
        behav_phenotype_name = 'log_' + behav_phenotype_name
    else:
        behav_phenotypes = np.array(behav_phenotypes)[:,None]
    
    #########################################################
    subjects_for_long_prediction = np.array(subjects_for_long_prediction)
    unique_subjects = np.unique(subjects_for_long_prediction)
    visits_per_unique_subject = np.zeros(unique_subjects.shape[0])
    subject_inds = np.zeros(subjects_for_long_prediction.shape[0])
    for i, temp_subj in enumerate(unique_subjects):
        subject_inds[subjects_for_long_prediction == temp_subj] = i
        visits_per_unique_subject[i] = np.sum(subjects_for_long_prediction == temp_subj)
    one_visit_subject_inds = np.where(visits_per_unique_subject == 1)
    two_visit_subject_inds = np.where(visits_per_unique_subject == 2)
    thr_visit_subject_inds = np.where(visits_per_unique_subject == 3)

    #print('subject_inds: {}'.format(subject_inds.shape))
    ########################################
    ########################################
    
    
    ######################################
    #####Newer with mixed effects#########
    ######################################
    df = {}
    pcs_text = ''
    for i in range(num_pcs):
        df['PC_{}'.format(i)] = pcs[:,i].squeeze()
        pcs_text = pcs_text + 'PC_{} + '.format(i)
    for i in [1,2,3]:
        df['visit_{}'.format(i)] = np.zeros(visits.shape[0])
        df['visit_{}'.format(i)][visits == i] = 1

    df[behav_phenotype_name] = behav_phenotypes.squeeze()
    df['age'] = ages.squeeze()
    df['subject_id'] = subjects_for_long_prediction.squeeze()

    Y, X_Full = dmatrices('{} ~ {} age + visit_2 + visit_3'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
    Y, X_Partial = dmatrices('{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
    model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
    results_full = model_full.fit(reml=False)
    model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, df['subject_id'])
    results_partial = model_partial.fit(reml=False)
    
    dof_difference = X_Full.shape[1] - X_Partial.shape[1]
    reduced_ll = results_partial.llf
    full_ll = results_full.llf
    LR_statistic = -2*(reduced_ll-full_ll)

    p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)
    print(results_full.summary())
    print('LikelihoodRatioStatistic: {}'.format(LR_statistic))
    print('P-Value: {}'.format(p_val))
    if isnone == False:
        with open(os.path.join(output_folder, 'pca.txt'), 'w') as f:
            f.write('Full Model: \n\n')
            f.write(results_full.summary().as_text())
            f.write('\n\n\n\nPartial Model:\n\n')
            f.write(results_partial.summary().as_text())
            f.write('\n\nLRT: {}, P-Value: {}'.format(LR_statistic, p_val))
    #########################################
    #########################################
                
        
    ######################################################################
    ######################################################################
    ######################################################################
    #Now ROI based approach (has been updated for mixed effects modeling)
    isc_pvals = np.zeros(hariri_isc_mats.shape[1])
    isc_tvals = np.zeros(hariri_isc_mats.shape[1])
    isc_coefs = np.zeros(hariri_isc_mats.shape[1])
    
    df = {}
    for j in [1,2,3]:
        df['visit_{}'.format(j)] = np.zeros(visits.shape[0])
        df['visit_{}'.format(j)][visits == j] = 1
    df[behav_phenotype_name] = behav_phenotypes.squeeze()
    df['age'] = ages.squeeze()
    df['subject_id'] = subjects_for_long_prediction.squeeze()
    
    Y, X_Partial = dmatrices('{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
    model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, df['subject_id'])
    results_partial = model_partial.fit(reml=False)
    reduced_residuals = results_partial.resid.values
    reduced_predictions = results_partial.fittedvalues.values

    for i in range(isc_pvals.shape[0]):
        
        df['image_data'] = iscs_for_phenotype_prediction[:,i].squeeze()
        Y, X_Full = dmatrices('{} ~ image_data + age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
        model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
        results_full = model_full.fit(reml=False)
        
        dof_difference = 1
        reduced_ll = results_partial.llf
        full_ll = results_full.llf
        LR_statistic = -2*(reduced_ll-full_ll)
        p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)

        isc_pvals[i] = -1*np.log10(p_val)
        isc_tvals[i] = results_full.tvalues['image_data']
        
        if i == 186:
            print('BE SURE THE SECOND COEFFICIENT IN THIS TABLE IS FOR IMAGING')
            print(results_full.summary())

    cifti_output_name = 'hariri_isc_with_{}_neglog10p_mixedeffects_uncorrected.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(isc_pvals[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    #cifti_output_name = 'hariri_isc_with_{}_coefs.pscalar.nii'.format(behav_phenotype_name)
    #make_pscalar_cifti(isc_coefs[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

    cifti_output_name = 'hariri_isc_with_{}_tvalues_mixedeffects_uncorrected.pscalar.nii'.format(behav_phenotype_name)
    make_pscalar_cifti(isc_tvals[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
    
    
    plt.figure(dpi = 150)
    plt.hist(isc_pvals, bins = 20)
    plt.title('Distribution of Second-Level Stats Across Regions ({})'.format(behav_phenotype_name))
    plt.ylabel('Count')
    plt.xlabel('-log10(p-value)')
    
    if isnone == False:
        plt.savefig(os.path.join(output_folder, 'dist_second_level_stats_across_regions.png'))
        
        
    #Marginal predictions/residuals for the previously fit reduced model without imaging
    original_reduced_marginal_predictions = results_partial.predict(X_Partial).values
    original_reduced_marginal_residuals = original_reduced_marginal_predictions - Y.values.squeeze()
    
    ########################################################################################
    ########################################################################################
    ########################################################################################
    null_dist = np.zeros(num_perms)
    temp_null_dist = np.zeros(hariri_isc_mats.shape[1])

    for i in range(num_perms):
        
        ##############################
        #####UPDATED FOR MIXED EFFECTS
        
        #reduced_residuals = results_partial.resid.values
        #reduced_predictions = results_partial.predict(X_Partial).values
        #subject_inds
        
        #print(reduced_residuals.shape)
        #print(np.max(subject_inds))
        #print(subject_inds.shape)
        
        permuted_phenotype_data = np.zeros(subject_inds.shape)
        permuted_subject_inds = np.zeros(unique_subjects.shape[0])
        permuted_subject_inds[visits_per_unique_subject == 1] = np.random.permutation(one_visit_subject_inds[0])
        permuted_subject_inds[visits_per_unique_subject == 2] = np.random.permutation(two_visit_subject_inds[0])
        permuted_subject_inds[visits_per_unique_subject == 3] = np.random.permutation(thr_visit_subject_inds[0])
        for j in range(len(unique_subjects)):

            single_subject_inds = np.where(subject_inds == permuted_subject_inds[j])
            shuffled_single_subject_inds = np.random.permutation(single_subject_inds[0])
            permuted_phenotype_data[np.where(subject_inds == j)[0]] = original_reduced_marginal_residuals[single_subject_inds] + \
                                                            original_reduced_marginal_predictions[np.where(subject_inds == j)[0]]
        
        ##############################
        ##############################
        
        if np.mod(i, 50) == 0:
            print(i)
            if isnone == False:
                with open(os.path.join(output_folder, 'perm_status.txt'), 'a') as f:
                    f.write('{}\n'.format(i))
            
        ####Refitting reduced model with permuted pheotype data
        df[behav_phenotype_name] = permuted_phenotype_data.squeeze()
        #Y, X_Partial = dmatrices('{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
        #model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, df['subject_id'])
        #results_partial = model_partial.fit(reml=False)
        #print('Fitted Partial Shuffled')
        #print(results_partial.summary())


        #Fit modle for each ROI (updated for mixed effects)
        for j in range(isc_pvals.shape[0]):
            
            df['image_data'] = iscs_for_phenotype_prediction[:,j].squeeze()
            Y, X_Full = dmatrices('{} ~ image_data + age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
            #X_Full = np.hstack([iscs_for_phenotype_prediction[:,j][:,None], ages, constant])
            #X_Partial = np.hstack([ages, constant])
            model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
            results_full = model_full.fit(reml=False)
            #print(results_full.summary())
            
            dof_difference = 1
            #reduced_ll = results_partial.llf
            #full_ll = results_full.llf
            #LR_statistic = -2*(reduced_ll-full_ll)
            #p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)
            #null_isc_pvals[j] = -1*np.log10(p_val)
            temp_null_dist[j] = np.absolute(results_full.tvalues['image_data'])

        #Save the best statistic (neglog10(p)) found across ROIs for the current permutation
        null_dist[i] = np.max(temp_null_dist)
        
    ###########################################################################################
    ###########################################################################################
    
    alpha_5th_percentile_cutoff = np.percentile(null_dist, 95)
    print("5th percentile cutoff (abs(t-stat)): {}".format(alpha_5th_percentile_cutoff))
    surviving_inds = np.where(np.absolute(isc_tvals) > alpha_5th_percentile_cutoff)[0]
    print("Regions surviving corrections at alpha = 0.05: {}".format(regions[surviving_inds]))
    print("Inds surviving correction at alpha = 0.05: {}".format(surviving_inds))
    pvals = []
    for temp_ind in surviving_inds:
        pvals.append(np.sum(null_dist > np.absolute(isc_tvals[temp_ind]))/(num_perms + 1))
    print("Corrected P-Vals of Inds surviving correction at alpha = 0.05: {}".format(pvals))

    ###Update this to work for arbritrary ROIs.....
    plt.figure(dpi = 150)
    plt.hist(null_dist, bins = 200)
    plt.title('Null Distribution')
    plt.ylabel('Count')
    plt.xlabel('test-statistic')
    plt.axvline(isc_pvals[186], linestyle = '--', color = 'red')
    plt.legend(['17Networks_LH_DefaultA_pCunPCC_2', 'Null Distribution'])
    
    if isnone == False:
        with open(os.path.join(output_folder, 'perm_details.txt'), 'w') as f:
            f.write("5th percentile cutoff (abs(test-statistic)): {}\n".format(alpha_5th_percentile_cutoff))
            f.write("Regions surviving corrections at alpha = 0.05: {}\n".format(regions[surviving_inds]))
            f.write("Inds surviving correction at alpha = 0.05: {}\n".format(surviving_inds))
            f.write("Corrected P-Vals of Inds surviving correction at alpha = 0.05: {}\n".format(pvals))
            np.save(os.path.join(output_folder, 'null_dist_abs_tstats.npy'), null_dist)
            np.save(os.path.join(output_folder, 'uncorrected_2nd_level_tstats.npy'), isc_tvals)

    

### Compilation of Mean TS across subjects

In [ ]:
####Calculate group average time series from all data
task_name = 'HARIRI'
conn_mats = glob.glob(os.path.join(input_dir, 'sub-*', 'ses-*', '*{}*.npy'.format(task_name)))  
mean_fd_threshold = 0.5

task_len = np.load(conn_mats[0]).shape[1]
num_rois = np.load(conn_mats[0]).shape[0]

###########################################################################
#First calculate the average timeseries using data from all available scans
###########################################################################
mean_ts = np.zeros((num_rois, task_len))
count = 0
all_mean_fds = []
all_max_fds = []
all_mean_dvars = []
all_max_dvars = []
all_runs_timeseries_dict = {}
for i, temp_mat in enumerate(conn_mats):
    splits = temp_mat.split('/')[-3:]
    temp_mat = np.load(conn_mats[i])
    if count == 0:
        print(temp_mat.shape)
    if temp_mat.shape[1] == task_len:
        
        temp_nuisance_file = os.path.join(conn_mats[i].replace(input_dir, nuisance_files_dir).split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
        df = pd.read_csv(temp_nuisance_file)
        if np.mean(df['fd_power']) > mean_fd_threshold:
            print('Skipping High Motion Scan')
            continue #dont include this run in processing
        all_mean_fds.append(np.mean(df['fd_power']))
        all_max_fds.append(np.max(df['fd_power']))
        all_mean_dvars.append(np.mean(df['dvars_white_matter']))
        all_max_dvars.append(np.max(df['dvars_white_matter']))
        
        count += 1
        all_runs_timeseries_dict[conn_mats[i]] = temp_mat.transpose()
        mean_ts = mean_ts + temp_mat
        if np.mod(i, 100) == 0:
            print(i)
all_runs_mean_ts = mean_ts/count
print('Number of runs used for mean generation: {}'.format(count))
num_total_runs = count

######################################################################################################
####Calculate first session ISC for all subjects (relating individual subject activation to group average)
######################################################################################################
subjects = glob.glob(os.path.join(input_dir, 'sub-*'))

count = 0

isc_subjects = []
isc_sessions = []
mean_fds = []
max_fds = []
mean_dvars = []
max_dvars = []

#Check which subjects have full data
for temp_subject in subjects:
    
    os.chdir(os.path.join(input_dir, temp_subject))
    conn_mats = glob.glob(os.path.join('ses-*', '*{}*.npy'.format(task_name))) 
    conn_mats.sort()
    if len(conn_mats):
        
        #Grab nuisance stats
        temp_nuisance_file = os.path.join(nuisance_files_dir, temp_subject.split('/')[-1], conn_mats[0].split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
        df = pd.read_csv(temp_nuisance_file)
        if np.mean(df['fd_power']) > mean_fd_threshold:
            continue #dont include this run in processing
        mean_fds.append(np.mean(df['fd_power']))
        max_fds.append(np.max(df['fd_power']))
        mean_dvars.append(np.mean(df['dvars_white_matter']))
        max_dvars.append(np.max(df['dvars_white_matter']))
        ####################
        
        isc_subjects.append(temp_subject)
        isc_sessions.append(conn_mats[0].split('/')[0])
        
        temp_mat = conn_mats[0]
        splits = temp_mat.split('/')[-3:]
        temp_mat = np.load(temp_mat)
        if count == 0:
            print(temp_mat.shape)
        if temp_mat.shape[1] == task_len:
            count += 1

hariri_isc_mats = np.zeros((count, num_rois))

count = 0
for i, temp_subject in enumerate(isc_subjects):
    
    os.chdir(os.path.join(input_dir, temp_subject))
    conn_mats = glob.glob(os.path.join('ses-*', '*{}*.npy'.format(task_name))) 
    conn_mats.sort()
    temp_mat = conn_mats[0]
    
    #Grab nuisance stats
    temp_nuisance_file = os.path.join(nuisance_files_dir, temp_subject.split('/')[-1], conn_mats[0].split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
    df = pd.read_csv(temp_nuisance_file)
    if np.mean(df['fd_power']) > mean_fd_threshold:
        continue #dont include this run in processing
    mean_fds.append(np.mean(df['fd_power']))
    max_fds.append(np.max(df['fd_power']))
    mean_dvars.append(np.mean(df['dvars_white_matter']))
    max_dvars.append(np.max(df['dvars_white_matter']))
    ####################
    
    splits = temp_mat.split('/')[-3:]
    temp_mat = np.load(temp_mat)
    if temp_mat.shape[1] == task_len:

        for j in range(num_rois):
            hariri_isc_mats[count,j] = np.corrcoef(temp_mat[j,:], all_runs_mean_ts[j,:])[0,1]
        count += 1
        if np.mod(i, 100) == 0:
            print(i)
print('Number of session 1 runs used (eigenvalues will be calculated for these): {}'.format(count))

  
######################################################################################################
####Calculate ISC for all subjects AND all sessions (relating individual subject activation to group average)
####We do this here for both tracking longitudinal trajectories + using as much data as possible for
####estimation of eigenvectors
######################################################################################################
count = 0
conn_mats = glob.glob(os.path.join(input_dir, 'sub-*', 'ses-*', '*{}*.npy'.format(task_name)))  
all_hariri_isc_mats = np.zeros((num_total_runs, hariri_isc_mats.shape[1]))
all_isc_subjects = []
all_isc_sessions = []
max_dvars = []
for i, temp_mat in enumerate(conn_mats):
    
    temp_nuisance_file = os.path.join(conn_mats[i].replace(input_dir, nuisance_files_dir).split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
    df = pd.read_csv(temp_nuisance_file)
    if np.mean(df['fd_power']) > mean_fd_threshold:
        continue #dont include this run in processing
    all_isc_subjects.append(temp_mat.split('/')[-3])
    all_isc_sessions.append(temp_mat.split('/')[-2])
    temp_mat = np.load(temp_mat)
    if temp_mat.shape[1] == task_len:

        for j in range(num_rois):
            all_hariri_isc_mats[count,j] = np.corrcoef(temp_mat[j,:], all_runs_mean_ts[j,:])[0,1]
        count += 1
        if np.mod(i, 100) == 0:
            print(i)
print('Number of {} runs that will be used to determine eigenvectors: {}'.format(task_name, count))

### Load rest runs so we can create "null" ISC maps used to determine how many Hariri ISCs to keep

In [ ]:
#Find all rest runs from any subject in the study whether/not they have hariri
task_name = 'rest'
conn_mats = glob.glob(os.path.join(input_dir, 'sub-*', 'ses-*', '*{}*.npy'.format(task_name)))  


hariri_len = 490 #After having removed first 4 hariri vols
task_len = np.load(conn_mats[0]).shape[1]
num_rois = np.load(conn_mats[0]).shape[0]
all_rest_runs_timeseries_dict = {}

#Construct the mean time signal for each ROI using the
#same number of runs that we have for HARIRI
mean_ts = np.zeros((num_rois, hariri_len))
count = 0
for i, temp_mat in enumerate(conn_mats):
    
    temp_nuisance_file = temp_nuisance_file = os.path.join(conn_mats[i].replace(input_dir, nuisance_files_dir).split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
    df = pd.read_csv(temp_nuisance_file)
    if np.mean(df['fd_power'].values[:hariri_len]) > mean_fd_threshold:
        print('Too much motion (1)')
        continue #dont include this run in processing
    
    splits = temp_mat.split('/')[-3:]
    temp_mat = np.load(conn_mats[i])
    if count == 0:
        print('Original {} dimensions: {}'.format(task_name, temp_mat.shape))
    if temp_mat.shape[1] == task_len:
        mean_ts = mean_ts + temp_mat[:,:hariri_len]
        count += 1
        if np.mod(i, 100) == 0:
            print(i)
    if count == num_total_runs:
        break
rest_mean_ts = mean_ts/count

#Construct the ISC maps for all of those runs
rest_isc_mats = np.zeros((num_total_runs, num_rois))
count = 0
for i, temp_mat in enumerate(conn_mats):
    temp_nuisance_file = temp_nuisance_file = os.path.join(conn_mats[i].replace(input_dir, nuisance_files_dir).split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
    df = pd.read_csv(temp_nuisance_file)
    if np.mean(df['fd_power'].values[:hariri_len]) > mean_fd_threshold:
        print('Too much motion (2)')
    splits = temp_mat.split('/')[-3:]
    temp_mat = np.load(conn_mats[i])
    all_rest_runs_timeseries_dict[conn_mats[i]] = temp_mat[:,:hariri_len].transpose()
    if temp_mat.shape[1] == task_len:
        
        for j in range(num_rois):
            rest_isc_mats[count,j] = np.corrcoef(temp_mat[j,:hariri_len], mean_ts[j,:hariri_len])[0,1]
        count += 1
        if np.mod(i, 100) == 0:
            print(i)
            
    if count == num_total_runs:
        break
print('Number of {} runs that will be used to determine eigenvectors: {}'.format(task_name, count))

### Calculate ISC PCs and compare rest/hariri PCs

In [ ]:
hariri_isc_pca = PCA(n_components=20)
#hariri_isc_pca.fit(hariri_isc_mats)
hariri_isc_pca.fit(all_hariri_isc_mats)
hariri_pc_scores = hariri_isc_pca.transform(hariri_isc_mats)
all_hariri_pc_scores = hariri_isc_pca.transform(all_hariri_isc_mats)

rest_isc_pca = PCA(n_components=20)
rest_isc_pca.fit(rest_isc_mats)
num_hariri_greater_than_rest = np.where((hariri_isc_pca.explained_variance_ - rest_isc_pca.explained_variance_[0]) > 0)[0].shape[0]
print('Number of HARIRI eigenvectors that explain more variance than rest eignevectors (for equivelant scan length): {}'.format(num_hariri_greater_than_rest))

plt.figure(dpi = 150)
plt.plot(hariri_isc_pca.explained_variance_)
plt.plot(rest_isc_pca.explained_variance_)
plt.axhline(rest_isc_pca.explained_variance_[0], c = 'black', linestyle = '--')
plt.legend(['Emotion Face-Matching', 'REST', 'Null Threshold'])
plt.xticks(np.arange(20))
plt.ylabel('Variance Explained by PC')
plt.title('Strength of PCs in Emotion Face-Matching vs. (Trimmed) Rest')
plt.xlabel('PC Number')

### Make plot showing similarity of group-average time courses across different areas during Hariri

In [ ]:
mean_ts_corr = np.corrcoef(all_runs_mean_ts)
network_ids = imagesc_schaeffer_17(mean_ts_corr, regions, [-0.9,0.9], border_width = 15, title = '', dpi = 300)

### Plot some of the group average time courses

In [ ]:
plt.figure(dpi = 250, figsize=(5,1))
plt.plot(all_runs_mean_ts[32,:])
plt.plot(all_runs_mean_ts[232,:])

plt.xlabel('Time (Repetition Number)')
plt.ylabel('Signal (a.u.)')
plt.legend(['LH-VisCent-Node', 'RH-VisCent-Node'], loc='upper right')
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)

plt.figure(dpi = 250, figsize = (4,0.7))
plt.hist(np.mean(all_hariri_isc_mats, axis = 0), bins = 30)
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.ylabel('Count')
plt.xlabel('Average Corr. Coef.', fontsize='small')


## Look at the distribution of correlations across region average time courses

In [ ]:
plt.figure(dpi=150)
upper = mean_ts_corr[np.triu_indices(mean_ts_corr.shape[0], k=1)]
fig = plt.hist(upper, bins = 400)
plt.xlabel('Similarity of Activations (r)')
plt.ylabel('Count')

In [ ]:
save = False
if save:
    #Save HARIRI ISC PCs
    cifti_output_name = 'hariri_isc_pcs.pscalar.nii'
    make_pscalar_cifti(hariri_isc_pca.components_, cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

    #Save Rest ISC PCs
    cifti_output_name = 'rest_isc_pcs.pscalar.nii'
    make_pscalar_cifti(rest_isc_pca.components_, cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

    #Save Mean HARIRI ISC
    cifti_output_name = 'hariri_isc_mean.pscalar.nii'
    make_pscalar_cifti(np.mean(all_hariri_isc_mats, axis = 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

    #Save Mean Rest ISC
    cifti_output_name = 'rest_isc_mean.pscalar.nii'
    make_pscalar_cifti(np.mean(rest_isc_mats, axis = 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))


In [ ]:
make_network_bar_chart(regions, hariri_isc_pca.components_[0,:], xlabel = 'Mean ISC PC Weighting')

### Load Behavioral Data And Find Which Sessions to Use For Predictions

In [ ]:
phenotype_name = 'bssi_subtotal_1'
num_pcs = 3
log_transform = True
num_perms = 5
bdi_tscores, bdi_slopes = make_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms)

In [ ]:
phenotype_name = 'bdi_total_score'
num_pcs = 3
log_transform = False
num_perms = 5
bdi_tscores, bdi_slopes = make_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms)

In [ ]:
phenotype_name = 'bdi_total_score'
num_pcs = 3
log_transform = False
num_perms = 5
bdi_tscores, bdi_slopes = make_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms)

### Code for AnnaK Like Analyses

In [ ]:
import statsmodels.api as sm
from patsy import dmatrices

behav_df = pd.read_csv(csv_path, index_col = 0)
behav_phenotype_name = 'bdi_total_score'
behav_phenotype_name_unedited = behav_phenotype_name
log_transform_behav_phenotype = False
behav_phenotypes = []
ages = []
pcs = []
iscs_for_phenotype_prediction = []
phenotype_subj_paths = []
phenotype_sessions = []
num_pcs = 4
timeseries_for_phenotype_subjects = []
for i, temp_subject in enumerate(isc_subjects):
    
    int_session = int(isc_sessions[i].split('-')[-1])
    subset_df = behav_df[behav_df['session_id'] == int_session]
    if subset_df.shape[0] == 1:
        if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
            pcs.append(hariri_pc_scores[i,:num_pcs])
            ages.append(subset_df['age'].values[0])
            behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
            iscs_for_phenotype_prediction.append(hariri_isc_mats[i, :])
            phenotype_subj_paths.append(temp_subject)
            phenotype_sessions.append(isc_sessions[i])
            
            found = 0
            for temp_key in all_runs_timeseries_dict.keys():
                if (temp_subject in temp_key) and (isc_sessions[i] in temp_key):
                    found += 1
                    timeseries_for_phenotype_subjects.append(all_runs_timeseries_dict[temp_key])
            if found != 1:
                raise NameError('Error: something is wrong with code')
            
iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
pcs = np.array(pcs)
constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
ages = np.array(ages)[:,None]

if log_transform_behav_phenotype:
    behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
    behav_phenotype_name = 'log_' + behav_phenotype_name
else:
    behav_phenotypes = np.array(behav_phenotypes)[:,None]
    
df = {}
pcs_text = ''
for i in range(num_pcs):
    df['PC_{}'.format(i)] = pcs[:,i]
    pcs_text = pcs_text + 'PC_{} + '.format(i)
    
df[behav_phenotype_name] = behav_phenotypes
df['age'] = ages


Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.OLS(Y,X_Full)
results_full = model_full.fit()

results_partial = sm.OLS(Y, X_Partial).fit()
results_full.compare_f_test(results_partial)
baseline_predictions = results_full.predict(X_Full)


In [ ]:
#timeseries_for_phenotype_subjects, behav_phenotypes
behav_phenotypes_orders = np.argsort(behav_phenotypes.squeeze())
bahav_phenotypes_ordered = behav_phenotypes.squeeze()[behav_phenotypes_orders]
ordered_timeseries_for_phenotype_subjects = []
for temp_order in range(behav_phenotypes_orders.shape[0]):
    ordered_timeseries_for_phenotype_subjects.append(timeseries_for_phenotype_subjects[behav_phenotypes_orders[temp_order]])

In [ ]:
#How could we test a disengagement hypothesis. We are interested in knowing, do individuals
#at one end of the phenotypic spectrum have a unique set of activity pattenrs, meaning that
#maybe people on one end of the phenotypic spectrum respond during certain times and people
#who are on the other end of the phenotypic spectrum respond during other times? Or could
#this be better described as disengagement. Meaning - the task resulted in an entrainment
#of activity patterns on one end of the biological spectrum but not the other?
#
#
#To ask this, we may think that (1) individuals at the high end of the phenotypic spectrum are
#more related to each other than others at the low end of the phenotpyic spectrum, but
#also (2) individuals on the low end of the phenotypic spectrum are more related to individuals
#on the high end of the phenotypic spectrum than to other individuals on the low end of the phenotypic
#spectrum. 
roi_ind = 186
single_subjects_iscs_phenotypes = np.zeros((bahav_phenotypes_ordered.shape[0],bahav_phenotypes_ordered.shape[0]))
for i in range(bahav_phenotypes_ordered.shape[0]):
    for j in range(bahav_phenotypes_ordered.shape[0]):
        single_subjects_iscs_phenotypes[i,j] = np.corrcoef(ordered_timeseries_for_phenotype_subjects[i][:,roi_ind],ordered_timeseries_for_phenotype_subjects[j][:,roi_ind])[0,1]
#bahav_phenotypes_ordered.shape
plt.figure(dpi=150)
plt.imshow(single_subjects_iscs_phenotypes, vmin = -0.1, vmax = 0.25)
plt.colorbar()
plt.xticks([])
plt.yticks([])
#plt.axvline(10.5, color='r', linewidth=0.75)
#plt.axhline(10.5, color='r', linewidth=0.75)

### Plot DMN PCC Time courses for high/low BDI

In [ ]:
roi_ind = 186
single_subjects_iscs_phenotypes = np.zeros((bahav_phenotypes_ordered.shape[0],bahav_phenotypes_ordered.shape[0]))
group_1_ts = np.zeros(490)
group_2_ts = np.zeros(490)
group_1 = 0
group_2 = 0

for i in range(bahav_phenotypes_ordered.shape[0]):
    if np.round(i/bahav_phenotypes_ordered.shape[0]) == 0:
        group_1_ts = group_1_ts + ordered_timeseries_for_phenotype_subjects[i][:,roi_ind]
        group_1 += 1
    else:
        group_2_ts = group_2_ts + ordered_timeseries_for_phenotype_subjects[i][:,roi_ind]
        group_2 += 1
        
group_1_ts = group_1_ts/group_1
group_2_ts = group_2_ts/group_2

skip_vols = 4
TR = 0.8
plt.figure(dpi = 300)
networks_to_image = [17]
plt.plot(group_1_ts, c = 'silver', linestyle = 'dashed')
plt.plot(group_2_ts, c = 'black')
plt.plot([30/TR - skip_vols,60/TR - skip_vols], [1.15,1.15], color='blue') #Shape
plt.plot([60/TR - skip_vols,90/TR - skip_vols], [1.15,1.15], color='red') #Emotion
plt.plot([90/TR - skip_vols,120/TR - skip_vols], [1.15,1.15], color='blue') #Shape
plt.plot([120/TR - skip_vols,150/TR - skip_vols], [1.15,1.15], color='red') #Emotion
plt.plot([150/TR - skip_vols,180/TR - skip_vols], [1.15,1.15], color='blue') #Shape
plt.plot([210/TR - skip_vols,240/TR - skip_vols], [1.15,1.15], color='red') #Emotion
plt.plot([240/TR - skip_vols,270/TR - skip_vols], [1.15,1.15], color='blue') #Shape
plt.plot([270/TR - skip_vols,300/TR - skip_vols], [1.15,1.15], color='red') #Emotion
plt.plot([300/TR - skip_vols,330/TR - skip_vols], [1.15,1.15], color='blue') #Shape
plt.plot([330/TR - skip_vols,360/TR - skip_vols], [1.15, 1.15], color='red') #Emotion
plt.legend(['Low BDI', 'High BDI', 'Shape', 'Emotion'])
#plt.xlim(200,300)
plt.ylabel('Activation (z-score)')
plt.xlabel('Time (Repitition Number)')
plt.ylim(-1.0,1.2)

### Calculate within block ISC (Note: we still are planning on computing the average signal based on equal weighting of each portion of a subject's scan, but outside of that the ISC will have weighting based on a specific regressors timing)

In [ ]:
from nilearn.glm.first_level import make_first_level_design_matrix

TR = 0.8
skip_vols = 4
offset = TR*skip_vols
frame_times = np.arange(mean_ts.shape[1]) * TR
hrf_model = "spm"

conditions = ['shape', 'emotion', 'shape', 'emotion', 'shape', 'emotion', 'shape', 'emotion', 'shape', 'emotion']
duration = [30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
onsets = [30 - offset, 60 - offset, 90 - offset, 120 - offset, 150 - offset, 210 - offset, 240 - offset, 270 - offset, 300 - offset, 330 - offset]
events = pd.DataFrame({"trial_type": conditions,
                      "onset": onsets,
                      "duration": duration})

X1 = make_first_level_design_matrix(
    frame_times,
    events,
    drift_model="polynomial",
    drift_order=0,
    hrf_model=hrf_model,
)

In [ ]:
b, a = construct_filter('lowpass', [0.25], 0.8, 3)
emotion_stim_filtered = apply_filter(b,a,X1['emotion'].copy())
shape_stim_filtered = apply_filter(b,a,X1['shape'])

In [ ]:
plt.figure(dpi=200)
plt.plot(shape_stim_filtered)
plt.plot(emotion_stim_filtered)
plt.legend(['Shape HRF', 'Emotion HRF'])
plt.xlabel('Time (TR#)')
plt.ylabel('Intensity (a.u.)')

In [ ]:
plt.figure(dpi=200, figsize=(5,2.5))

absolute_weighting = np.absolute(shape_stim_filtered)
sum_normalized_weighting_arr = absolute_weighting/np.sum(absolute_weighting)
plt.plot(sum_normalized_weighting_arr)

absolute_weighting = np.absolute(emotion_stim_filtered)
sum_normalized_weighting_arr = absolute_weighting/np.sum(absolute_weighting)
plt.plot(sum_normalized_weighting_arr)
plt.legend(['Shape Weighting', 'Emotion Weighting'])
plt.xlabel('Time (TR#)')
plt.ylabel('Weighting Density')

In [ ]:
####Calculate group average time series from all data
task_name = 'HARIRI'
conn_mats = glob.glob(os.path.join(input_dir, 'sub-*', 'ses-*', '*{}*.npy'.format(task_name)))  
mean_fd_threshold = 0.5

task_len = np.load(conn_mats[0]).shape[1]
num_rois = np.load(conn_mats[0]).shape[0]

######################################################################################################
####Calculate first session ISC for all subjects (relating individual subject activation to group average)
######################################################################################################
subjects = glob.glob(os.path.join(input_dir, 'sub-*'))

count = 0

isc_subjects = []
isc_sessions = []
mean_fds = []
max_fds = []
mean_dvars = []
max_dvars = []

#Check which subjects have full data
for temp_subject in subjects:
    
    os.chdir(os.path.join(input_dir, temp_subject))
    conn_mats = glob.glob(os.path.join('ses-*', '*{}*.npy'.format(task_name))) 
    conn_mats.sort()
    if len(conn_mats):
        
        #Grab nuisance stats
        temp_nuisance_file = os.path.join(nuisance_files_dir, temp_subject.split('/')[-1], conn_mats[0].split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
        df = pd.read_csv(temp_nuisance_file)
        if np.mean(df['fd_power']) > mean_fd_threshold:
            continue #dont include this run in processing
        mean_fds.append(np.mean(df['fd_power']))
        max_fds.append(np.max(df['fd_power']))
        mean_dvars.append(np.mean(df['dvars_white_matter']))
        max_dvars.append(np.max(df['dvars_white_matter']))
        ####################
        
        isc_subjects.append(temp_subject)
        isc_sessions.append(conn_mats[0].split('/')[0])
        
        temp_mat = conn_mats[0]
        splits = temp_mat.split('/')[-3:]
        temp_mat = np.load(temp_mat)
        if count == 0:
            print(temp_mat.shape)
        if temp_mat.shape[1] == task_len:
            count += 1


hariri_isc_block_mats = np.zeros((count, 2, num_rois))
count = 0
for i, temp_subject in enumerate(isc_subjects):
    
    os.chdir(os.path.join(input_dir, temp_subject))
    conn_mats = glob.glob(os.path.join('ses-*', '*{}*.npy'.format(task_name))) 
    conn_mats.sort()
    temp_mat = conn_mats[0]
    
    #Grab nuisance stats
    temp_nuisance_file = os.path.join(nuisance_files_dir, temp_subject.split('/')[-1], conn_mats[0].split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
    df = pd.read_csv(temp_nuisance_file)
    if np.mean(df['fd_power']) > mean_fd_threshold:
        continue #dont include this run in processing
    mean_fds.append(np.mean(df['fd_power']))
    max_fds.append(np.max(df['fd_power']))
    mean_dvars.append(np.mean(df['dvars_white_matter']))
    max_dvars.append(np.max(df['dvars_white_matter']))
    ####################
    
    splits = temp_mat.split('/')[-3:]
    temp_mat = np.load(temp_mat)
    if temp_mat.shape[1] == task_len:

        for j in range(num_rois):
            hariri_isc_block_mats[count,0,j] = calc_weighted_corr_coef(temp_mat[j,:], all_runs_mean_ts[j,:], emotion_stim_filtered)
            hariri_isc_block_mats[count,1,j] = calc_weighted_corr_coef(temp_mat[j,:], all_runs_mean_ts[j,:], shape_stim_filtered)
        count += 1
        if np.mod(i, 100) == 0:
            print(i)
print('Number of session 1 runs used (eigenvalues will be calculated for these): {}'.format(count))

  
######################################################################################################
####Calculate ISC for all subjects AND all sessions (relating individual subject activation to group average)
####We do this here for both tracking longitudinal trajectories + using as much data as possible for
####estimation of eigenvectors
######################################################################################################
count = 0
conn_mats = glob.glob(os.path.join(input_dir, 'sub-*', 'ses-*', '*{}*.npy'.format(task_name)))  
all_hariri_isc_block_mats = np.zeros((num_total_runs, 2, hariri_isc_mats.shape[1]))
all_isc_subjects = []
all_isc_sessions = []
max_dvars = []
for i, temp_mat in enumerate(conn_mats):
    
    temp_nuisance_file = os.path.join(conn_mats[i].replace(input_dir, nuisance_files_dir).split('_bold_Atlas_MSMAll_2_d40_WRN_hp2000_clean_GSR.ptseries_SkipVolsRemoved_FilterApplied_ZNormalized.npy')[0] + '_bold', 'nuisance_signals.csv')
    df = pd.read_csv(temp_nuisance_file)
    if np.mean(df['fd_power']) > mean_fd_threshold:
        continue #dont include this run in processing
    all_isc_subjects.append(temp_mat.split('/')[-3])
    all_isc_sessions.append(temp_mat.split('/')[-2])
    temp_mat = np.load(temp_mat)
    if temp_mat.shape[1] == task_len:

        for j in range(num_rois):
            all_hariri_isc_block_mats[count,0,j] = calc_weighted_corr_coef(temp_mat[j,:], all_runs_mean_ts[j,:], emotion_stim_filtered)
            all_hariri_isc_block_mats[count,1,j] = calc_weighted_corr_coef(temp_mat[j,:], all_runs_mean_ts[j,:], shape_stim_filtered)
        count += 1
        if np.mod(i, 100) == 0:
            print(i)
print('Number of {} runs that will be used to determine eigenvectors: {}'.format(task_name, count))

In [ ]:
#Save Mean HARIRI Emotion ISC
cifti_output_name = 'hariri_emotion_isc_mean.pscalar.nii'
make_pscalar_cifti(np.mean(all_hariri_isc_block_mats[:,0,:], axis = 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

#Save Mean HARIRI Shape ISC
cifti_output_name = 'hariri_shape_isc_mean.pscalar.nii'
make_pscalar_cifti(np.mean(all_hariri_isc_block_mats[:,1,:], axis = 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

In [ ]:
behav_df = pd.read_csv(csv_path, index_col = 0)
behav_phenotype_name = 'bdi_total_score'
behav_phenotype_name_unedited = behav_phenotype_name
log_transform_behav_phenotype = False
behav_phenotypes = []
ages = []
pcs = []
iscs_for_phenotype_prediction = []
within_emotion_isc_for_phenotype_prediction = []
within_shape_isc_for_phenotype_prediction = []
phenotype_subj_paths = []
phenotype_sessions = []
timeseries_for_phenotype_subjects = []
for i, temp_subject in enumerate(isc_subjects):

    int_session = int(isc_sessions[i].split('-')[-1])
    subset_df = behav_df[behav_df['session_id'] == int_session]
    if subset_df.shape[0] == 1:
        if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
            pcs.append(hariri_pc_scores[i,:num_pcs])
            ages.append(subset_df['age'].values[0])
            behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
            iscs_for_phenotype_prediction.append(hariri_isc_mats[i, :])
            within_emotion_isc_for_phenotype_prediction.append(hariri_isc_block_mats[i,0,:].squeeze())
            within_shape_isc_for_phenotype_prediction.append(hariri_isc_block_mats[i,1,:].squeeze())
            phenotype_subj_paths.append(temp_subject)
            phenotype_sessions.append(isc_sessions[i])

            found = 0
            for temp_key in all_runs_timeseries_dict.keys():
                if (temp_subject in temp_key) and (isc_sessions[i] in temp_key):
                    found += 1
                    timeseries_for_phenotype_subjects.append(all_runs_timeseries_dict[temp_key])
            if found != 1:
                raise NameError('Error: something is wrong with code')

iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
pcs = np.array(pcs)
constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
ages = np.array(ages)[:,None]

roi_num = 186
roi_emotion_vals = np.array(within_emotion_isc_for_phenotype_prediction)[:,roi_num]
roi_shape_vals = np.array(within_shape_isc_for_phenotype_prediction)[:,roi_num]

if log_transform_behav_phenotype:
    behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
    behav_phenotype_name = 'log_' + behav_phenotype_name
else:
    behav_phenotypes = np.array(behav_phenotypes)[:,None]

########################################
########################################
########################################
########################################
########################################
########################################
########################################
########################################
###Check how things look with the unweighted ISC
df = {}
df[behav_phenotype_name] = behav_phenotypes
df['age'] = ages
df['dmn_pcc_unweighted'] = iscs_for_phenotype_prediction[:,roi_num]


Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, 'dmn_pcc_unweighted + '), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.OLS(Y,X_Full)
results_full = model_full.fit()

print('FULL UNWEIGHTED_ISC')
results_partial = sm.OLS(Y, X_Partial).fit()
print('Test for {} comparing full/reduced model: f = {}, p = {}'.format(behav_phenotype_name, results_full.compare_f_test(results_partial)[0], results_full.compare_f_test(results_partial)[1]))
baseline_predictions = results_full.predict(X_Full)
print('Full model results summary: \n\n')
print(results_full.summary())
    
    
########################################
########################################
########################################
########################################
########################################
########################################
########################################
########################################
###First check with emotion blocks
df = {}
df[behav_phenotype_name] = behav_phenotypes
df['age'] = ages
df['dmn_pcc_emotion'] = roi_emotion_vals

print('EMOTION WEIGHTED ISC')
Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, 'dmn_pcc_emotion + '), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.OLS(Y,X_Full)
results_full = model_full.fit()

results_partial = sm.OLS(Y, X_Partial).fit()
print('Test for {} comparing full/reduced model: f = {}, p = {}'.format(behav_phenotype_name, results_full.compare_f_test(results_partial)[0], results_full.compare_f_test(results_partial)[1]))
baseline_predictions = results_full.predict(X_Full)
print('Full model results summary: \n\n')
print(results_full.summary())

########################################
########################################
########################################
########################################
########################################
########################################
########################################
########################################
###Second check shape blocks
print('\n\n\n')
df = {}
df[behav_phenotype_name] = behav_phenotypes
df['age'] = ages
df['dmn_pcc_shape'] = roi_shape_vals


Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, 'dmn_pcc_shape + '), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.OLS(Y,X_Full)
results_full = model_full.fit()

print('SHAPE WEIGHTED ISC')
results_partial = sm.OLS(Y, X_Partial).fit()
print('Test for {} comparing full/reduced model: f = {}, p = {}'.format(behav_phenotype_name, results_full.compare_f_test(results_partial)[0], results_full.compare_f_test(results_partial)[1]))
baseline_predictions = results_full.predict(X_Full)
print('Full model results summary: \n\n')
print(results_full.summary())


########################################
########################################
########################################
########################################
########################################
########################################
########################################
########################################
###Then check emotion + shape together
print('\n\n\n')
df = {}
df[behav_phenotype_name] = behav_phenotypes
df['age'] = ages
df['dmn_pcc_shape'] = roi_shape_vals
df['dmn_pcc_emotion'] = roi_emotion_vals
df['dmn_pcc_unweighted'] = iscs_for_phenotype_prediction[:,roi_num]

print('EMOTION AND SHAPE WEIGHTED ISC')
Y, X_Full = dmatrices('{} ~ {} age'.format(behav_phenotype_name, 'dmn_pcc_shape + dmn_pcc_emotion + dmn_pcc_unweighted + '), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.OLS(Y,X_Full)
results_full = model_full.fit()

results_partial = sm.OLS(Y, X_Partial).fit()
print('Test for {} comparing full/reduced model: f = {}, p = {}'.format(behav_phenotype_name, results_full.compare_f_test(results_partial)[0], results_full.compare_f_test(results_partial)[1]))
baseline_predictions = results_full.predict(X_Full)
print('Full model results summary: \n\n')
print(results_full.summary())

In [ ]:
plt.figure(dpi=200, figsize=(5,2.5))
plt.bar(['No Weighting', 'Emotion Weighting', 'Shape Weighting'], [-4.217, -3.821, -2.701], color=['black', 'red', 'blue'])
plt.ylabel('Test Statistic')
plt.title('Relation Between Precuneus DMN Node ISC-Types and BDI', fontsize ='10')
#plt.

# Now run normal task activation type analysis

In [ ]:
first_session_task_activations_t = np.zeros(hariri_isc_mats.shape)
all_hariri_task_activations_t = np.zeros((len(all_runs_timeseries_dict.keys()), hariri_isc_mats.shape[1]))
all_rest_runs_task_activations_t = np.zeros((len(all_rest_runs_timeseries_dict.keys()), hariri_isc_mats.shape[1]))
first_session_task_activations_coefdif = np.zeros(hariri_isc_mats.shape)
all_hariri_task_activations_coefdif = np.zeros((len(all_runs_timeseries_dict.keys()), hariri_isc_mats.shape[1]))
all_rest_runs_task_activations_coefdif = np.zeros((len(all_rest_runs_timeseries_dict.keys()), hariri_isc_mats.shape[1]))
task_contrast = np.array([0, 1, -1]) #This is right as of 8/25, but if changes are made be sure this still lines up correctly with the regression
task_df = {'emotion': emotion_stim_filtered, 'shape': shape_stim_filtered}

#All Rest Runs
for i, temp_run in enumerate(all_rest_runs_timeseries_dict.keys()):
    temp_subject_data = all_rest_runs_timeseries_dict[temp_run]
    for j in range(hariri_isc_mats.shape[1]):
        
        task_df['signal'] = temp_subject_data[:,j]
        Y, X = dmatrices('signal ~ 1 + emotion + shape', data=task_df, return_type='dataframe')
        task_model = sm.OLS(Y,X)
        task_results = task_model.fit()
        contrast_results = task_results.t_test(task_contrast)
        all_rest_runs_task_activations_t[i,j] = contrast_results.tvalue[0,0]
        all_rest_runs_task_activations_coefdif[i,j] = task_results.params[1] - task_results.params[2]
        #raise NameError('pause')
print('Done with Rest Runs')
        
#All Hariri Runs
for i, temp_run in enumerate(all_runs_timeseries_dict.keys()):
    temp_subject_data = all_runs_timeseries_dict[temp_run]
    for j in range(hariri_isc_mats.shape[1]):
        
        task_df['signal'] = temp_subject_data[:,j]
        Y, X = dmatrices('signal ~ 1 + emotion + shape', data=task_df, return_type='dataframe')
        task_model = sm.OLS(Y,X)
        task_results = task_model.fit()
        contrast_results = task_results.t_test(task_contrast)
        all_hariri_task_activations_t[i,j] = contrast_results.tvalue[0,0]
        all_hariri_task_activations_coefdif[i,j] = task_results.params[1] - task_results.params[2]
print('Done with Hariri Runs (1)')

#Hariri Session 1 Runs (shhhh I know it is wasteful...)
for i in range(hariri_isc_mats.shape[0]):
    for temp_key in all_runs_timeseries_dict.keys():
        if (isc_sessions[i] in temp_key) and (isc_subjects[i] in temp_key):
            temp_subject_data = all_runs_timeseries_dict[temp_key]
            continue
    for j in range(hariri_isc_mats.shape[1]):
        
        task_df['signal'] = temp_subject_data[:,j]
        Y, X = dmatrices('signal ~ 1 + emotion + shape', data=task_df, return_type='dataframe')
        task_model = sm.OLS(Y,X)
        task_results = task_model.fit()
        contrast_results = task_results.t_test(task_contrast)
        first_session_task_activations_t[i,j] = contrast_results.tvalue[0,0]
        first_session_task_activations_coefdif[i,j] = task_results.params[1] - task_results.params[2]
print('Done with Hariri Runs (2)')

In [ ]:
print(task_results.summary())

In [ ]:
task_results.params[1] - task_results.params[2]

In [ ]:
hariri_task_pca = PCA(n_components=20)
#hariri_isc_pca.fit(hariri_isc_mats)
hariri_task_pca.fit(all_hariri_task_activations_coefdif)
hariri_task_pc_scores = hariri_task_pca.transform(first_session_task_activations_coefdif)
all_hariri_task_pc_scores = hariri_task_pca.transform(all_hariri_task_activations_coefdif)

rest_task_pca = PCA(n_components=20)
rest_task_pca.fit(all_rest_runs_task_activations_coefdif)
num_task_hariri_greater_than_rest = np.where((hariri_task_pca.explained_variance_ - rest_task_pca.explained_variance_[0]) > 0)[0].shape[0]
print('Number of Task (Emotion > Shape) HARIRI eigenvectors that explain more variance than rest eignevectors (for equivelant scan length): {}'.format(num_task_hariri_greater_than_rest))

plt.figure(dpi = 150)
plt.plot(hariri_task_pca.explained_variance_)
plt.plot(rest_task_pca.explained_variance_)
plt.axhline(rest_task_pca.explained_variance_[0], c = 'black', linestyle = '--')
plt.legend(['HARIRI', 'REST', 'Null Threshold'])
plt.xticks(np.arange(20))
plt.ylabel('Variance Explained by PC')
plt.title('Strength of Task (Emotion - Shape) PCs in Emotion Face-Matching vs. (Trimmed) Rest')
plt.xlabel('PC Number')

In [ ]:
plt.figure(dpi=150)
plt.hist(np.mean(first_session_task_activations_coefdif, axis= 0), bins=40)
plt.xlabel('Emotion - Shape (a.u.)')
plt.ylabel('Count (Number of Regions)')
plt.title('Distribution of Contrast Strength Across All ROIs')
#Save HARIRI Task Activation Maps
cifti_output_name = 'hariri_first_session_emotion_GreaterThan_shape_tstat.pscalar.nii'
make_pscalar_cifti(np.mean(first_session_task_activations_t, axis= 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
cifti_output_name = 'hariri_first_session_emotion_GreaterThan_shape_coefdif.pscalar.nii'
make_pscalar_cifti(np.mean(first_session_task_activations_coefdif, axis= 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))
#Save HARIRI Task PCs
cifti_output_name = 'hariri_emotion_GreaterThan_shape_PCs.pscalar.nii'
make_pscalar_cifti(hariri_task_pca.components_, cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

#first_session_task_activations_coefdif
cifti_output_name = 'hariri_emotion_GreaterThan_shape_coefdif.pscalar.nii'
make_pscalar_cifti(np.mean(all_hariri_task_activations_coefdif, axis= 0)[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

In [ ]:
behav_df = pd.read_csv(csv_path, index_col = 0)
behav_measure = 'bdi_total_score'
ages = []
pcs = []
iscs_for_phenotype_prediction = []
num_pcs = 3
num_coef_dif_pcs = 2
pcs_dict = {}
ages_dict = {}
iscs_dict = {}
behavs_dict = {}
ics_dict = {}
coef_difs_dict = {}
coef_difs_pc_scores_dict = {}
sessions_dict = {}


for i, temp_subject in enumerate(isc_subjects):
    
    temp_subject = temp_subject.split('/')[-1]
    temp_session_inds = np.where(np.array(all_isc_subjects) == temp_subject)
    temp_sessions = np.array(all_isc_sessions)[temp_session_inds]
    temp_pc_scores = np.zeros((len(temp_sessions), num_pcs))
    temp_ages = np.zeros(len(temp_sessions))
    temp_phens = np.zeros(len(temp_sessions))
    temp_coef_dif = np.zeros((len(temp_sessions), num_rois))
    temp_ses = np.zeros((len(temp_sessions)))
    temp_coef_dif_pc_scores = np.zeros((len(temp_sessions), num_coef_dif_pcs))
    temp_iscs = np.zeros((len(temp_sessions), num_rois))
    for j, temp_session in enumerate(temp_sessions):
        int_session = int(temp_session.split('-')[-1])
        subset_df = behav_df[behav_df['session_id'] == int_session]
        if subset_df.shape[0] == 1:
            if (np.isnan(subset_df['age'].values[0]) == False):
                temp_ages[j] = subset_df['age'].values[0]
                temp_pc_scores[j,:] = all_hariri_pc_scores[temp_session_inds[0][j],:num_pcs]
                temp_coef_dif[j, :] = all_hariri_task_activations_coefdif[temp_session_inds[0][j],:]
                temp_coef_dif_pc_scores[j,:] = all_hariri_task_pc_scores[temp_session_inds[0][j],:num_coef_dif_pcs] #This was wrong before
                temp_iscs[j,:] = all_hariri_isc_mats[temp_session_inds[0][j],:]
                temp_ses[j] = int(subset_df.index.values[0].split('_')[1])
                
                #iscs_for_phenotype_prediction.append(hariri_isc_mats[i, :])
            else:
                temp_ages[j] = np.nan
                temp_ses[j] = np.nan
        elif subset_df.shape[0] == 0:
            temp_ages[j] = np.nan
            temp_ses[j] = np.nan
        else:
            raise NameError('Error 2')
    pcs_dict[temp_subject] = temp_pc_scores
    iscs_dict[temp_subject] = temp_iscs
    ages_dict[temp_subject] = temp_ages
    coef_difs_dict[temp_subject] = temp_coef_dif
    coef_difs_pc_scores_dict[temp_subject] = temp_coef_dif_pc_scores
    sessions_dict[temp_subject] = temp_ses
        
    

In [ ]:
phenotype_name = 'bdi_total_score'
num_pcs = 2
log_transform = False
num_perms = 1
#make_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms)
make_behav_predictions_task(hariri_task_pc_scores, first_session_task_activations_coefdif, csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms)

## Mixed Effects Longitudinal Modeling

In [ ]:
behav_df = pd.read_csv(csv_path, index_col = 0)
behav_phenotype_name = 'sitbi_nssie_lastyear_freq'
log_transform_behav_phenotype = True
behav_phenotypes = []
ages = []
pcs = []
visits = []
iscs_for_phenotype_prediction = []
subjects_for_long_prediction = []
num_pcs = 3

for i, temp_subject in enumerate(all_isc_subjects):
    
    int_session = int(all_isc_sessions[i].split('-')[-1])
    subset_df = behav_df[behav_df['session_id'] == int_session]
    if subset_df.shape[0] == 1:
        if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
            pcs.append(all_hariri_pc_scores[i,:num_pcs])
            ages.append(subset_df['age'].values[0])
            behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
            iscs_for_phenotype_prediction.append(all_hariri_isc_mats[i, :])
            subjects_for_long_prediction.append(temp_subject)
            visits.append(int(subset_df.index.values[0].split('_')[1]))
            
iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
pcs = np.array(pcs)
constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
ages = np.array(ages)[:,None]
visits = np.array(visits)

if log_transform_behav_phenotype:
    behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
    behav_phenotype_name = 'log_' + behav_phenotype_name
else:
    behav_phenotypes = np.array(behav_phenotypes)[:,None]

In [ ]:
df = {}
pcs_text = ''
for i in range(num_pcs):
    df['PC_{}'.format(i)] = pcs[:,i].squeeze()
    pcs_text = pcs_text + 'PC_{} + '.format(i)
for i in [1,2,3]:
    df['visit_{}'.format(i)] = np.zeros(visits.shape[0])
    df['visit_{}'.format(i)][visits == i] = 1
    
df[behav_phenotype_name] = behav_phenotypes.squeeze()
df['age'] = ages.squeeze()
df['subject_id'] = np.array(subjects_for_long_prediction).squeeze()

Y, X_Full = dmatrices('{} ~ {} age + visit_2 + visit_3'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
results_full = model_full.fit(reml=False)
model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, subjects_for_long_prediction)
results_partial = model_partial.fit(reml=False)

reduced_residuals = results_partial.resid.values
reduced_predictions = results_partial.fittedvalues.values
full_predictions = results_full.fittedvalues.values
full_predictions_imaging_only = np.zeros(full_predictions.shape[0])
for temp_pc in range(num_pcs):
    full_predictions_imaging_only = full_predictions_imaging_only + X_Full['PC_{}'.format(temp_pc)]*results_full.params['PC_{}'.format(temp_pc)]


plt.figure(dpi=200)
plt.scatter(full_predictions_imaging_only, Y.values)
plt.xlabel('Predicted {} - Predicted Non-Imaging Effect'.format(behav_phenotype_name))
plt.ylabel('Actual {}'.format(behav_phenotype_name))
print(np.corrcoef(full_predictions_imaging_only, Y.values.squeeze()))


haufe_transform_pc_covs_mixed = np.zeros(hariri_isc_mats.shape[1])
for temp_region in range(hariri_isc_mats.shape[1]):
    haufe_transform_pc_covs_mixed[temp_region] = np.cov(iscs_for_phenotype_prediction[:,temp_region].squeeze(), full_predictions)[0,1]
make_network_bar_chart(regions, haufe_transform_pc_covs_mixed, xlabel = 'Haufe-Transformed Effect Size (Cov)')
cifti_output_name = 'hariri_isc_with_{}_HaufeTransformedPCModelEffects_Covariances_MixedEffects.pscalar.nii'.format(behav_phenotype_name)
make_pscalar_cifti(haufe_transform_pc_covs_mixed[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

In [ ]:
haufe_transform_pc_covs_mixed = np.zeros(hariri_isc_mats.shape[1])
for temp_region in range(hariri_isc_mats.shape[1]):
    haufe_transform_pc_covs_mixed[temp_region] = np.cov(iscs_for_phenotype_prediction[:,temp_region].squeeze(), full_predictions)[0,1]
make_network_bar_chart(regions, haufe_transform_pc_covs_mixed, xlabel = 'Haufe-Transformed Effect Size (Cov)')
cifti_output_name = 'hariri_isc_with_{}_HaufeTransformedPCModelEffects_Covariances_MixedEffects.pscalar.nii'.format(behav_phenotype_name)
make_pscalar_cifti(haufe_transform_pc_covs_mixed[None,:], cifti_reference_path, os.path.join(cifti_output_folder, cifti_output_name))

In [ ]:
dof_difference = 3
reduced_ll = results_partial.llf
full_ll = results_full.llf
LR_statistic = -2*(reduced_ll-full_ll)

p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)
print('LikelihoodRatioStatistic: {}'.format(LR_statistic))
print('P-Value: {}'.format(p_val))

### Redoing Permutations

In [ ]:
behav_df = pd.read_csv(csv_path, index_col = 0)
behav_phenotype_name = 'bdi_total_score'
log_transform_behav_phenotype = False
behav_phenotypes = []
ages = []
pcs = []
visits = []
iscs_for_phenotype_prediction = []
subjects_for_long_prediction = []
num_pcs = 3

for i, temp_subject in enumerate(all_isc_subjects):
    
    int_session = int(all_isc_sessions[i].split('-')[-1])
    subset_df = behav_df[behav_df['session_id'] == int_session]
    if subset_df.shape[0] == 1:
        if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
            pcs.append(all_hariri_pc_scores[i,:num_pcs])
            ages.append(subset_df['age'].values[0])
            behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
            iscs_for_phenotype_prediction.append(all_hariri_isc_mats[i, :])
            subjects_for_long_prediction.append(temp_subject)
            visits.append(int(subset_df.index.values[0].split('_')[1]))
            
iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
pcs = np.array(pcs)
constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
ages = np.array(ages)[:,None]
visits = np.array(visits)

if log_transform_behav_phenotype:
    behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
    behav_phenotype_name = 'log_' + behav_phenotype_name
else:
    behav_phenotypes = np.array(behav_phenotypes)[:,None]
    
###################################################################
###################################################################

df = {}
pcs_text = ''
for i in range(num_pcs):
    df['PC_{}'.format(i)] = pcs[:,i].squeeze()
    pcs_text = pcs_text + 'PC_{} + '.format(i)
for i in [1,2,3]:
    df['visit_{}'.format(i)] = np.zeros(visits.shape[0])
    df['visit_{}'.format(i)][visits == i] = 1
    
df[behav_phenotype_name] = behav_phenotypes.squeeze()
df['age'] = ages.squeeze()
df['subject_id'] = np.array(subjects_for_long_prediction).squeeze()

original_reduced_model_text = '{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name)
Y, X_Partial = dmatrices(original_reduced_model_text, data=df, return_type='dataframe')
model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, df['subject_id'])
original_results_partial = model_partial.fit(reml=False)
print(original_reduced_model_text)

original_full_model_text = '{} ~ {} age + visit_2 + visit_3'.format(behav_phenotype_name, pcs_text)
Y, X_Full = dmatrices(original_full_model_text, data=df, return_type='dataframe')
model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
original_results_full = model_full.fit(reml=False)
print(original_full_model_text)


###################################################################
###################################################################



original_LR_statistic = -2*(original_results_partial.llf - original_results_full.llf)
original_reduced_residuals = original_results_partial.resid.values
original_reduced_predictions = original_results_partial.fittedvalues.values

subjects_for_perm = np.array(subjects_for_long_prediction).squeeze()
unique_subjects = np.unique(subjects_for_perm)
subject_inds = np.zeros(subjects_for_perm.shape[0])
for i, temp_subj in enumerate(unique_subjects):
    subject_inds[subjects_for_perm == temp_subj] = i
    
num_perms = 100
null_dist = np.zeros(num_perms)
for i in range(num_perms):
    
    permuted_phenotype_data = np.zeros(subject_inds.shape)
    for j in range(int(np.max(subject_inds))):

        reduced_residuals_temp_inds = np.where(subject_inds == j)
        shuffled = np.random.permutation(reduced_residuals_temp_inds[0])
        #permuted_phenotype_data[reduced_residuals_temp_inds] = original_reduced_residuals[shuffled] + original_full_predictions_minus_general_intercept[reduced_residuals_temp_inds]
        #permuted_phenotype_data[reduced_residuals_temp_inds] = original_reduced_residuals[shuffled] + original_reduced_predictions[reduced_residuals_temp_inds]
        permuted_phenotype_data[reduced_residuals_temp_inds] = original_results_partial.resid.values[shuffled] + original_results_partial.params[0] + \
                                                                original_results_partial.params[1]*df['age'][reduced_residuals_temp_inds] + \
                                                                original_results_partial.params[2]*df['visit_2'][reduced_residuals_temp_inds] + \
                                                                original_results_partial.params[3]*df['visit_3'][reduced_residuals_temp_inds] + \
                                                                original_results_partial.random_effects[unique_subjects[j]].values
        #print(reduced_residuals_temp_inds)
    df['shuffled_phenotype'] = permuted_phenotype_data
    
    ###Partial
    shuffled_reduced_model_text = 'shuffled_phenotype ~ age + visit_2 + visit_3'
    Y, X_Perm_Partial = dmatrices(shuffled_reduced_model_text, data=df, return_type='dataframe')
    if i == 0:
        print(shuffled_reduced_model_text)
    model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Perm_Partial, df['subject_id'])
    results_partial = model_partial.fit(reml=False)
    reduced_ll = results_partial.llf
    
    #Full
    shuffled_full_model_text = 'shuffled_phenotype ~ PC_0 + PC_1 + PC_2 + age + visit_2 + visit_3'
    Y, X_Perm_Full = dmatrices(shuffled_full_model_text, data=df, return_type='dataframe')
    if i == 0:
        print(shuffled_full_model_text)
    model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Perm_Full, df['subject_id'])
    results_full = model_full.fit(reml=False)
    full_ll = results_full.llf

    dof_difference = 3
    LR_statistic = -2*(reduced_ll-full_ll)
    #p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)
    null_dist[i] = LR_statistic
    
    if np.mod(i,200) == 0:
        print(i)

In [ ]:
original_reduced_model_text = '{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name)
Y, X_Partial = dmatrices(original_reduced_model_text, data=df, return_type='dataframe')
model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, df['subject_id'])
results_partial = model_partial.fit(reml=False)
print(original_reduced_model_text)
print(results_partial.params)

#Hopefully this recovers predicted values
predicted_phenotype = np.zeros(results_full.fittedvalues.shape[0])
for j in range(int(np.max(subject_inds))):

    reduced_residuals_temp_inds = np.where(subject_inds == j)
    shuffled = np.random.permutation(reduced_residuals_temp_inds[0])
    #permuted_phenotype_data[reduced_residuals_temp_inds] = original_reduced_residuals[shuffled] + original_full_predictions_minus_general_intercept[reduced_residuals_temp_inds]
    #permuted_phenotype_data[reduced_residuals_temp_inds] = original_reduced_residuals[shuffled] + original_reduced_predictions[reduced_residuals_temp_inds]
    predicted_phenotype[reduced_residuals_temp_inds] = original_reduced_residuals[reduced_residuals_temp_inds] + results_partial.params[0] + \
                                                            results_partial.params[1]*df['age'][reduced_residuals_temp_inds] + \
                                                            results_partial.params[2]*df['visit_2'][reduced_residuals_temp_inds] + \
                                                            results_partial.params[3]*df['visit_3'][reduced_residuals_temp_inds] + \
                                                            results_partial.random_effects[unique_subjects[j]].values

In [ ]:
np.sum(original_LR_statistic < null_dist)/(null_dist.shape[0] + 1)

In [ ]:
Y, X_Full = dmatrices('{} ~ {} age + visit_2 + visit_3'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
results_full = model_full.fit(reml=False)
model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, subjects_for_long_prediction)
results_partial = model_partial.fit(reml=False)

reduced_residuals = results_partial.resid.values
reduced_predictions = results_partial.fittedvalues.values

subjects_for_perm = np.array(subjects_for_long_prediction).squeeze()
unique_subjects = np.unique(subjects_for_perm)
subject_inds = np.zeros(subjects_for_perm.shape[0])
for i, temp_subj in enumerate(unique_subjects):
    subject_inds[subjects_for_perm == temp_subj] = i
    
num_perms = 2000
null_dist = np.zeros(num_perms)
for i in range(num_perms):
    
    #permuted_phenotype_data = np.zeros(subject_inds.shape)
    #for j in range(int(np.max(subject_inds))):

    #    reduced_residuals_temp_inds = np.where(subject_inds == j)
    #    shuffled = np.random.permutation(reduced_residuals_temp_inds[0])
    #    permuted_phenotype_data[reduced_residuals_temp_inds] = reduced_residuals[shuffled] + reduced_predictions[reduced_residuals_temp_inds]
        
    df['shuffled_phenotype'] = np.random.normal(size=Y.shape)
    df['age'] = np.random.normal(size=Y.shape)
    df['PC_0'] = np.random.normal(size=Y.shape)
    df['PC_1'] = np.random.normal(size=Y.shape)
    df['PC_2'] = np.random.normal(size=Y.shape)
    Y, X_Perm_Partial = dmatrices('shuffled_phenotype ~ age + visit_2 + visit_3', data=df, return_type='dataframe')
    Y, X_Perm_Full = dmatrices('shuffled_phenotype ~ {} + age + visit_2 + visit_3'.format(pcs_text), data=df, return_type='dataframe')
    model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Perm_Partial, df['subject_id'])
    results_partial = model_partial.fit(reml=False)
    model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Perm_Full, df['subject_id'])
    results_full = model_full.fit(reml=False)

    dof_difference = num_pcs
    reduced_ll = results_partial.llf
    full_ll = results_full.llf
    LR_statistic = -2*(reduced_ll-full_ll)
    p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)
    null_dist[i] = p_val
    
    if np.mod(i,10) == 0:
        print(i)

In [ ]:
results_partial.resid.values
subjects_for_perm = np.array(subjects_for_long_prediction).squeeze()

In [ ]:
unique_subjects = np.unique(subjects_for_perm)
subject_inds = np.zeros(subjects_for_perm.shape[0])
for i, temp_subj in enumerate(unique_subjects):
    subject_inds[subjects_for_perm == temp_subj] = i
    

In [ ]:
partial_predictions = results_partial.predict(X_Partial).values
partial_predictions

## Now the Mixed Effects Modeling at the single ROI level with permutation-based multiple comparisons corrections

In [ ]:
phenotype_name = 'sitbi_nssie_lastyear_freq'
num_pcs = 3
log_transform = True
num_perms = 1

if log_transform:
    log_phen_name = 'log_' + phenotype_name
else:
    log_phen_name = phenotype_name
output_folder = mixed_effects_output_dir_template.format(log_phen_name)
make_mixed_effects_behav_predictions(csv_path, phenotype_name, log_transform, num_pcs, cifti_output_folder, num_perms, output_folder=output_folder)

### Now adding interaction terms

In [ ]:
behav_df = pd.read_csv(csv_path, index_col = 0)
behav_phenotype_name = 'sitbi_nssie_lastyear_freq'
log_transform_behav_phenotype = True
behav_phenotypes = []
ages = []
pcs = []
visits = []
iscs_for_phenotype_prediction = []
subjects_for_long_prediction = []
num_pcs = 3

for i, temp_subject in enumerate(all_isc_subjects):
    
    int_session = int(all_isc_sessions[i].split('-')[-1])
    subset_df = behav_df[behav_df['session_id'] == int_session]
    if subset_df.shape[0] == 1:
        if (np.isnan(subset_df[behav_phenotype_name].values[0]) == False) and ((subset_df[behav_phenotype_name].values[0]) >= 0) and (np.isnan(subset_df['age'].values[0]) == False):
            pcs.append(all_hariri_pc_scores[i,:num_pcs])
            ages.append(subset_df['age'].values[0])
            behav_phenotypes.append(subset_df[behav_phenotype_name].values[0])
            iscs_for_phenotype_prediction.append(all_hariri_isc_mats[i, :])
            subjects_for_long_prediction.append(temp_subject)
            visits.append(int(subset_df.index.values[0].split('_')[1]))
            
iscs_for_phenotype_prediction = np.array(iscs_for_phenotype_prediction)
pcs = np.array(pcs)
constant = np.ones(iscs_for_phenotype_prediction.shape[0])[:,None]
ages = np.array(ages)[:,None]
visits = np.array(visits)

if log_transform_behav_phenotype:
    behav_phenotypes = np.log(np.array(behav_phenotypes) + 1)[:,None]
    behav_phenotype_name = 'log_' + behav_phenotype_name
else:
    behav_phenotypes = np.array(behav_phenotypes)[:,None]

In [ ]:
df = {}
pcs_text = ''
for i in range(num_pcs):
    df['PC_{}'.format(i)] = pcs[:,i].squeeze()
    pcs_text = pcs_text + 'PC_{} + '.format(i)
for i in [1,2,3]:
    df['visit_{}'.format(i)] = np.zeros(visits.shape[0])
    df['visit_{}'.format(i)][visits == i] = 1
    
df[behav_phenotype_name] = behav_phenotypes.squeeze()
df['age'] = ages.squeeze()
df['subject_id'] = np.array(subjects_for_long_prediction).squeeze()

Y, X_Full = dmatrices('{} ~ PC_0 + PC_0:visit_2 + PC_0:visit_3 + PC_1 + PC_1:visit_2 + PC_1:visit_3 + PC_2 + PC_2:visit_2 + PC_2:visit_3 + age + visit_2 + visit_3'.format(behav_phenotype_name, pcs_text), data=df, return_type='dataframe')
Y, X_Partial = dmatrices('{} ~ PC_0 + PC_1 + PC_2 + age + visit_2 + visit_3'.format(behav_phenotype_name), data=df, return_type='dataframe')
model_full = sm.regression.mixed_linear_model.MixedLM(Y,X_Full, df['subject_id'])
results_full = model_full.fit(reml=False)
model_partial = sm.regression.mixed_linear_model.MixedLM(Y,X_Partial, subjects_for_long_prediction)
results_partial = model_partial.fit(reml=False)

In [ ]:
dof_difference = 6
reduced_ll = results_partial.llf
full_ll = results_full.llf
LR_statistic = -2*(reduced_ll-full_ll)

p_val = scipy.stats.chi2.sf(LR_statistic, dof_difference)
print('LikelihoodRatioStatistic: {}'.format(LR_statistic))
print('P-Value: {}'.format(p_val))